In [1]:
import pandas as pd
from pathlib import Path

from langchain_core.documents import Document

In [3]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "output"

DATASET_PATH = DATA_DIR / "hindcon_master_dataset.xlsx"

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATASET_PATH)
print("Dataset exists:", DATASET_PATH.exists())

Project root: c:\Users\User\Documents\Projects\TDS_Based_Product_Knowledge_Assitant
Dataset: c:\Users\User\Documents\Projects\TDS_Based_Product_Knowledge_Assitant\output\hindcon_master_dataset.xlsx
Dataset exists: True


In [4]:
df = pd.read_excel(DATASET_PATH)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 271
Columns: ['Product_ID', 'Category', 'Product_Name', 'TDS_Link', 'TDS_Text', 'TDS_Status', 'MSDS_Link', 'MSDS_Text', 'MSDS_Status']


In [6]:
df.head(10)

,Product_ID,Category,Product_Name,TDS_Link,TDS_Text,TDS_Status,MSDS_Link,MSDS_Text,MSDS_Status
0,HC001,Waterproofing Compounds,Hind Crystel Seal,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND CRYSTAL SEAL\nDESC...,Success,NaN,NaN,MSDS Link Missing
1,HC002,Waterproofing Compounds,Hind Crystel Seal (1),https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND CRYSTAL SEAL (I)\n...,Success,NaN,NaN,MSDS Link Missing
2,HC003,Waterproofing Compounds,Hind Plug - S,https://www.hindcon.com/wp-content/uploads/202...,HINDCONCHEMICALS LTD.\nHIND PLUG - S \nDESCRIP...,Success,NaN,NaN,MSDS Link Missing
3,HC004,Waterproofing Compounds,Hind Proof No – 1,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND PROOF NO-1 \nDESCR...,Success,NaN,NaN,MSDS Link Missing
4,HC005,Waterproofing Compounds,Hind Proof No – 2,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND PROOF NO-2\nDESCRI...,Success,NaN,NaN,MSDS Link Missing
5,HC006,Waterproofing Compounds,Hind Proof No – 3,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND PROOF NO-3\nDESCRI...,Success,NaN,NaN,MSDS Link Missing
6,HC007,Waterproofing Compounds,Hind Powder WP,https://www.hindcon.com/wp-content/uploads/202...,HINDCONCHEMICALS LTD.\nHIND POWDER WP\nDESCRIP...,Success,NaN,NaN,MSDS Link Missing
7,HC008,Concrete & Mortar Admixtures,Hind Flow Set 1000,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND FLOW SET 1000 \nDE...,Success,https://www.hindcon.com/wp-content/uploads/202...,"HINDCON\nCHEMICALS LTD.\n+91 33 24490839, 9830...",Success
8,HC009,Concrete & Mortar Admixtures,Hind Plast AEA,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND PLAST AEA\nDESCRIP...,Success,NaN,NaN,MSDS Link Missing
9,HC010,Concrete & Mortar Admixtures,Hind Plast AEP,https://www.hindcon.com/wp-content/uploads/202...,HINDCON\nCHEMICALS LTD.HIND PLAST AEP\nDESCRIP...,Success,NaN,NaN,MSDS Link Missing


In [7]:
df["TDS_Status"].value_counts(dropna=False)

TDS_Status
Success                                                                                                                    260
Error: Invalid URL 'TDS not found': No scheme supplied. Perhaps you meant https://TDS not found?                             6
Error: 404 Client Error: Not Found for url: https://www.hindcon.com/wp-content/uploads/2025/05/HIND-CURE-WB_TDS.pdf          1
Error: 404 Client Error: Not Found for url: https://www.hindcon.com/wp-content/uploads/2026/02/HIND-ELASTO-COAT-TDS.pdf      1
TDS Link Missing                                                                                                             1
Error: 404 Client Error: Not Found for url: https://www.hindcon.com/wp-content/uploads/2025/07/HIND-SET-LAF-100.pdf          1
Error: 404 Client Error: Not Found for url: https://www.hindcon.com/wp-content/uploads/2026/05/HIND-SEALANT-E.pdf            1
Name: count, dtype: int64

In [8]:
df["TDS_Text"].notna().sum()

np.int64(258)

In [9]:
REQUIRED_COLUMNS = [
    "Product_ID",
    "Category",
    "Product_Name",
    "TDS_Link",
    "TDS_Text",
    "TDS_Status"
]

missing_columns = [
    column for column in REQUIRED_COLUMNS
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Dataset validation successful.")

Dataset validation successful.


In [10]:
valid_tds = df[
    df["TDS_Text"].notna()
    & (df["TDS_Text"].astype(str).str.strip() != "")
    & (df["TDS_Status"].astype(str).str.lower() == "success")
].copy()

print("Total records:", len(df))
print("Valid TDS records:", len(valid_tds))

Total records: 271
Valid TDS records: 258


In [11]:
import re


def clean_tds_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Remove excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Remove excessive blank lines
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()

In [12]:
sample_text = valid_tds.iloc[0]["TDS_Text"]

cleaned_text = clean_tds_text(sample_text)

print(cleaned_text[:3000])

HINDCON
CHEMICALS LTD.HIND CRYSTAL SEAL
DESCRIPTION 
Hind Crystal Seal is a capillary waterprooﬁng formulation of proprietary blends of moisture active 
chemicals, quartz, sand and cement. The chemical contained in Hind Crystal Seal formulations 
requires the presence of moisture for the chemical reaction to take place and in combination with 
the “free lime”, the chemical in Hind Crystal Seal forms long chain complexes which crystallizes 
in the capillaries. It conforms to IS: 2645-2003 & ACI 212.3R - 10.
USES 
Hind Crystal Seal protects concrete and mortar making it 100% waterproof. It improves the 
adhesion between top coat and concrete or mortar. It is used for waterprooﬁng of water reservoir, 
water tower, tunnels, dams, canals, concrete pipes, water treatment plant, sea wall, bridges and 
decks etc.
ADVANTAGES
• Ready for use only by mixing with water.
• Can be applied on damp concrete surface with no standing water.
• Non-toxic and can be applied on Concrete storage tank used fo

In [13]:
documents = []

for _, row in valid_tds.iterrows():

    text = clean_tds_text(row["TDS_Text"])

    metadata = {
        "product_id": str(row["Product_ID"]),
        "product_name": str(row["Product_Name"]),
        "category": str(row["Category"]),
        "tds_link": str(row["TDS_Link"]),
        "tds_status": str(row["TDS_Status"]),
    }

    document = Document(
        page_content=text,
        metadata=metadata
    )

    documents.append(document)

In [14]:
print("LangChain documents:", len(documents))

LangChain documents: 258


In [15]:
print(documents[0].page_content[:2000])
print()
print(documents[0].metadata)

HINDCON
CHEMICALS LTD.HIND CRYSTAL SEAL
DESCRIPTION 
Hind Crystal Seal is a capillary waterprooﬁng formulation of proprietary blends of moisture active 
chemicals, quartz, sand and cement. The chemical contained in Hind Crystal Seal formulations 
requires the presence of moisture for the chemical reaction to take place and in combination with 
the “free lime”, the chemical in Hind Crystal Seal forms long chain complexes which crystallizes 
in the capillaries. It conforms to IS: 2645-2003 & ACI 212.3R - 10.
USES 
Hind Crystal Seal protects concrete and mortar making it 100% waterproof. It improves the 
adhesion between top coat and concrete or mortar. It is used for waterprooﬁng of water reservoir, 
water tower, tunnels, dams, canals, concrete pipes, water treatment plant, sea wall, bridges and 
decks etc.
ADVANTAGES
• Ready for use only by mixing with water.
• Can be applied on damp concrete surface with no standing water.
• Non-toxic and can be applied on Concrete storage tank used fo

In [16]:
from urllib.parse import urlparse
from pathlib import Path


def get_filename_from_url(url):
    if not isinstance(url, str) or not url.strip():
        return ""

    path = urlparse(url).path
    return Path(path).name

In [17]:
print(get_filename_from_url(
    valid_tds.iloc[0]["TDS_Link"]
))

HIND-CRYSTAL-SEAL_TDS-1.pdf


In [18]:
documents = []

for _, row in valid_tds.iterrows():

    text = clean_tds_text(row["TDS_Text"])

    metadata = {
        "product_id": str(row["Product_ID"]),
        "product_name": str(row["Product_Name"]),
        "category": str(row["Category"]),
        "tds_link": str(row["TDS_Link"]),
        "document_name": get_filename_from_url(row["TDS_Link"]),
        "tds_status": str(row["TDS_Status"]),
    }

    documents.append(
        Document(
            page_content=text,
            metadata=metadata
        )
    )

In [19]:
def validate_document(document: Document) -> list[str]:

    errors = []

    if not document.page_content.strip():
        errors.append("Empty document content")

    required_metadata = [
        "product_id",
        "product_name",
        "category",
        "document_name"
    ]

    for field in required_metadata:
        if not document.metadata.get(field):
            errors.append(f"Missing metadata: {field}")

    return errors

In [20]:
errors = validate_document(documents[0])

print(errors)

[]


In [21]:
validation_results = []

for doc in documents:

    errors = validate_document(doc)

    validation_results.append({
        "product_id": doc.metadata["product_id"],
        "product_name": doc.metadata["product_name"],
        "errors": errors
    })

failed = [
    result for result in validation_results
    if result["errors"]
]

print("Documents:", len(documents))
print("Failed validation:", len(failed))

Documents: 258
Failed validation: 0


In [22]:
SECTION_NAMES = [
    "DESCRIPTION",
    "USES",
    "ADVANTAGES",
    "APPLICATION",
    "PROPERTIES",
    "COVERAGE",
    "SHELF LIFE",
    "PACKING",
    "HANDLING PRECAUTION",
    "HANDLING PRECAUTIONS",
    "STORAGE",
    "TECHNICAL DATA",
    "SPECIFICATIONS",
    "LIMITATIONS",
]

In [23]:
def normalize_heading(text: str) -> str:
    text = text.strip().upper()
    text = re.sub(r"[^A-Z0-9\s/&-]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text


def is_section_heading(line: str) -> bool:
    normalized = normalize_heading(line)

    return (
        normalized in SECTION_NAMES
        or any(
            normalized.startswith(section + ":")
            for section in SECTION_NAMES
        )
    )

In [24]:
def extract_sections(text: str) -> list[dict]:
    lines = text.splitlines()

    sections = []
    current_section = "GENERAL"
    current_content = []

    for line in lines:
        line = line.strip()

        if not line:
            continue

        if is_section_heading(line):

            if current_content:
                sections.append({
                    "section": current_section,
                    "content": "\n".join(current_content).strip()
                })

            heading = normalize_heading(line)

            for section_name in SECTION_NAMES:
                if heading.startswith(section_name):
                    current_section = section_name
                    break

            current_content = []

        else:
            current_content.append(line)

    if current_content:
        sections.append({
            "section": current_section,
            "content": "\n".join(current_content).strip()
        })

    return sections

In [26]:
sample_doc = documents[130]

sections = extract_sections(sample_doc.page_content)

for section in sections:
    print("=" * 70)
    print("SECTION:", section["section"])
    print(section["content"][:1000])

SECTION: GENERAL
HINDCON
CHEMICALS LTD.
SECTION: DESCRIPTION
Hind Grout GP is a High Strength, Free Flowing Non-Shrink, Non-metallic and Chloride Free Grout conforming to BS-
1881, Part 116-1983.
SECTION: USES
Hind Grout GP is used for grouting of Bolt Pockets and Base plates grouting of all equipments, Columns in pre-cast
construction concrete, Girders, Beams, Repair & Rehabilitation of Structures.
SECTION: ADVANTAGES
Hind Grout GP is an economical and easy to use material, requiring only addition of water. It is very easy to mix and
placement can be done with high ﬂow characteristic, rapid strength development, adjustable consistency and
2
Compressive strength up to 45N/mm (28 days) is achieved. It does not segregate and bleed. It is impact and vibration
resistant, non-corrosive to steel and iron, non-toxic and safe to handle.
SECTION: APPLICATION
All grouting areas must be free from the loose concrete, oil, grease, and dust where Hind Grout GP will be placed. The
grouting areas to b

In [27]:
section_documents = []

for doc in documents:

    sections = extract_sections(doc.page_content)

    for section_data in sections:

        section = section_data["section"]
        content = section_data["content"]

        if not content.strip():
            continue

        metadata = dict(doc.metadata)

        metadata["section"] = section

        section_documents.append(
            Document(
                page_content=content,
                metadata=metadata
            )
        )

print("Original documents:", len(documents))
print("Section documents:", len(section_documents))

Original documents: 258
Section documents: 2304


In [28]:
for i, doc in enumerate(section_documents[:10]):

    print("\n" + "=" * 80)
    print("CHUNK:", i)
    print("Product:", doc.metadata["product_name"])
    print("Section:", doc.metadata["section"])
    print("-" * 80)
    print(doc.page_content[:800])


CHUNK: 0
Product: Hind Crystel Seal
Section: GENERAL
--------------------------------------------------------------------------------
HINDCON
CHEMICALS LTD.HIND CRYSTAL SEAL

CHUNK: 1
Product: Hind Crystel Seal
Section: DESCRIPTION
--------------------------------------------------------------------------------
Hind Crystal Seal is a capillary waterprooﬁng formulation of proprietary blends of moisture active
chemicals, quartz, sand and cement. The chemical contained in Hind Crystal Seal formulations
requires the presence of moisture for the chemical reaction to take place and in combination with
the “free lime”, the chemical in Hind Crystal Seal forms long chain complexes which crystallizes
in the capillaries. It conforms to IS: 2645-2003 & ACI 212.3R - 10.

CHUNK: 2
Product: Hind Crystel Seal
Section: USES
--------------------------------------------------------------------------------
Hind Crystal Seal protects concrete and mortar making it 100% waterproof. It improves the
adhesion 

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ". ",
        "; ",
        ", ",
        " "
    ]
)

In [31]:
final_chunks = []

for doc in section_documents:

    split_texts = text_splitter.split_text(
        doc.page_content
    )

    for chunk_number, chunk_text in enumerate(split_texts):

        metadata = dict(doc.metadata)

        metadata["chunk_id"] = chunk_number

        final_chunks.append(
            Document(
                page_content=chunk_text,
                metadata=metadata
            )
        )

print("Section documents:", len(section_documents))
print("Final chunks:", len(final_chunks))

Section documents: 2304
Final chunks: 2817


In [32]:
def validate_chunk(doc: Document) -> list[str]:

    errors = []

    if not doc.page_content.strip():
        errors.append("Empty chunk")

    required_metadata = [
        "product_id",
        "product_name",
        "section",
        "document_name",
    ]

    for field in required_metadata:

        if not doc.metadata.get(field):
            errors.append(
                f"Missing metadata: {field}"
            )

    return errors

In [33]:
chunk_validation = []

for doc in final_chunks:

    errors = validate_chunk(doc)

    chunk_validation.append({
        "product_id": doc.metadata.get("product_id"),
        "product_name": doc.metadata.get("product_name"),
        "section": doc.metadata.get("section"),
        "errors": errors
    })

failed_chunks = [
    item
    for item in chunk_validation
    if item["errors"]
]

print("Total chunks:", len(final_chunks))
print("Failed chunks:", len(failed_chunks))

Total chunks: 2817
Failed chunks: 0


In [34]:
technical_sections = [
    "PROPERTIES",
    "TECHNICAL DATA",
    "SPECIFICATIONS",
    "COVERAGE",
    "SHELF LIFE",
    "PACKING"
]

technical_chunks = [
    doc
    for doc in final_chunks
    if doc.metadata.get("section") in technical_sections
]

print("Technical chunks:", len(technical_chunks))

Technical chunks: 1002


In [35]:
for doc in technical_chunks[:10]:

    print("\n" + "=" * 80)
    print("Product:", doc.metadata["product_name"])
    print("Section:", doc.metadata["section"])
    print(doc.page_content)


Product: Hind Crystel Seal
Section: PROPERTIES
Aspect Grey Powder
Bulk density gm/cc 1.25
Water Penetration (BS EN 12390 Part 8:2000) Nil
Water Pressure Head (mtr) 40-50
Corrosion Resistance (pH) (mixed with water) 10 + 1
Particle Size (micron) 40-150
Heat Resistance (ASTM C 1064-86) Same as Concrete
Service Life Same as RCC

Product: Hind Crystel Seal
Section: COVERAGE
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.

Product: Hind Crystel Seal
Section: SHELF LIFE
Best before 12 months from the date of manufacture. Should be stored in a cool and dry place
o o
away from direct sunlight at temperature ranging between 2 C (Min.) and 45 C (Max.).

Product: Hind Crystel Seal
Section: PACKING
10 Kg. in HDPE Bag.

Product: Hind Crystel Seal (1)
Section: SHELF LIFE
Best before 12 months from the date of manufacture. Should be stored in a cool and dry place away
o o
from direct sunlight at temperature ranging between 2 C (Min.) and 45 C (Max.).

Product

In [36]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("GEMINI_API_KEY not found in .env")

print("API key loaded successfully.")

API key loaded successfully.


In [37]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [38]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

print("Embedding model initialized.")

Embedding model initialized.


In [39]:
from langchain_chroma import Chroma

In [40]:
VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore"

VECTORSTORE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Vector store:", VECTORSTORE_DIR)

Vector store: c:\Users\User\Documents\Projects\TDS_Based_Product_Knowledge_Assitant\vectorstore


In [41]:
vectorstore = Chroma(
    collection_name="hindcon_tds",
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR)
)

In [42]:
ids = []

for i, doc in enumerate(final_chunks):

    product_id = doc.metadata["product_id"]
    section = doc.metadata["section"]

    chunk_id = doc.metadata["chunk_id"]

    unique_id = (
        f"{product_id}_{section}_{chunk_id}_{i}"
    )

    ids.append(unique_id)

In [46]:
from langchain_huggingface import HuggingFaceEmbeddings

In [47]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Local embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Local embedding model loaded successfully.


In [48]:
from langchain_chroma import Chroma

VECTORSTORE_DIR = PROJECT_ROOT / "vectorstore"

vectorstore = Chroma(
    collection_name="hindcon_tds_local",
    embedding_function=embeddings,
    persist_directory=str(VECTORSTORE_DIR)
)

print("Chroma vector store created.")

Chroma vector store created.


In [49]:
ids = []

for i, doc in enumerate(final_chunks):

    product_id = doc.metadata["product_id"]
    section = doc.metadata["section"]
    chunk_id = doc.metadata["chunk_id"]

    unique_id = (
        f"{product_id}_{section}_{chunk_id}_{i}"
    )

    ids.append(unique_id)

print("IDs generated:", len(ids))

IDs generated: 2817


In [50]:
print("Chunks:", len(final_chunks))
print("IDs:", len(ids))

Chunks: 2817
IDs: 2817


In [51]:
vectorstore.add_documents(
    documents=final_chunks,
    ids=ids
)

print(f"Successfully added {len(final_chunks)} chunks.")

Successfully added 2817 chunks.


In [52]:
query = "What is the coverage of Hind Crystel Seal?"

results = vectorstore.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(results, start=1):

    print("=" * 80)
    print("RESULT:", i)
    print("Product:", doc.metadata.get("product_name"))
    print("Product ID:", doc.metadata.get("product_id"))
    print("Section:", doc.metadata.get("section"))
    print("-" * 80)
    print(doc.page_content)

RESULT: 1
Product: Hind PU Sealer 275
Product ID: HC211
Section: DESCRIPTION
--------------------------------------------------------------------------------
Hind PU Sealer 275 is a low modulus, two part pitch free polyurethane elastomeric pavement sealant formulated to
accommodate continual movement. It is resistant to fuel, oil, hydraulic ﬂuid and skydrol and remains in a stable
condition when exposed to high temperature and high humidity climatic conditions. It conforms to BS 5212 and US
Federal Speciﬁcation SS-S 200 E.
RESULT: 2
Product: Hind Sealant PU 160
Product ID: HC210
Section: APPLICATION
--------------------------------------------------------------------------------
IMPORTANT NOTICE :
HINDCONCHEMICALS LTD.
VISIT OUR WEBSITE
HIND SEALANT PU 160
SINGLE COMPONENT POLYURETHANE SEALANT
RESULT: 3
Product: Hind Crystel Seal
Product ID: HC001
Section: GENERAL
--------------------------------------------------------------------------------
HINDCON
CHEMICALS LTD.HIND CRYSTAL SEAL
RE

In [53]:
query = "What is the setting time of Hind Plug - S?"

results = vectorstore.similarity_search(
    query,
    k=5
)

for doc in results:
    print(
        doc.metadata["product_name"],
        "|",
        doc.metadata["section"]
    )

Hind Plug - S | DESCRIPTION
Hind Plug - S | DESCRIPTION
Hind Plug - S | DESCRIPTION
Hind Plug - S | GENERAL
Hind Shot Set P | PROPERTIES


In [54]:
query = "What is the shelf life of Hind Crystel Seal?"

results = vectorstore.similarity_search(
    query,
    k=5
)

for doc in results:
    print(
        doc.metadata["product_name"],
        "|",
        doc.metadata["section"]
    )

Hind Silicone Sealant WP 200 | PROPERTIES
Hind Plast Super Anti Freeze | SHELF LIFE
Hind EP Prime 800 | PACKING
Hind Coat EP | SHELF LIFE
Hind Selfloor 2000 | SHELF LIFE


In [55]:
for doc in final_chunks:
    text = doc.page_content

    if "Hind Plug - S" in text and "Hind Shot Set P" in text:
        print("=" * 100)
        print("PRODUCT:", doc.metadata.get("product_name"))
        print("PRODUCT ID:", doc.metadata.get("product_id"))
        print("SECTION:", doc.metadata.get("section"))
        print(text[:3000])

In [57]:
product_names = (
    valid_tds["Product_Name"]
    .dropna()
    .astype(str)
    .tolist()
)

def find_products_in_text(text):
    found = []

    for product in product_names:
        if product.lower() in text.lower():
            found.append(product)

    return found


mixed_chunks = []

for doc in final_chunks:
    found = find_products_in_text(doc.page_content)

    if len(found) > 1:
        mixed_chunks.append({
            "chunk_id": doc.metadata.get("chunk_id"),
            "product_id": doc.metadata.get("product_id"),
            "product_name": doc.metadata.get("product_name"),
            "section": doc.metadata.get("section"),
            "products_found": found
        })

print("Mixed chunks:", len(mixed_chunks))

Mixed chunks: 503


In [58]:
pd.DataFrame(mixed_chunks).head(20)

,chunk_id,product_id,product_name,section,products_found
0,0,HC013,Hind Plast Super SCA-800,GENERAL,"[Hind Plast Super SCA-800, Hind Plast Super SC..."
1,0,HC013,Hind Plast Super SCA-800,DESCRIPTION,"[Hind Plast Super SCA-800, Hind Plast Super SC..."
2,0,HC013,Hind Plast Super SCA-800,USES,"[Hind Plast Super SCA-800, Hind Plast Super SC..."
3,0,HC013,Hind Plast Super SCA-800,HANDLING PRECAUTION,"[Hind Plast Super SCA-800, Hind Plast Super SC..."
4,1,HC013,Hind Plast Super SCA-800,HANDLING PRECAUTION,"[Hind Plast Super SCA-800, Hind Plast Super SC..."
5,0,HC015,Hind Plast Super 160 (P),GENERAL,"[Hind Plast Super 160 (P), Hind Plast Super 16..."
6,0,HC015,Hind Plast Super 160 (P),DESCRIPTION,"[Hind Plast Super 160 (P), Hind Plast Super 16..."
7,0,HC015,Hind Plast Super 160 (P),USES,"[Hind Plast Super 160 (P), Hind Plast Super 16..."
8,0,HC015,Hind Plast Super 160 (P),APPLICATION,"[Hind Plast Super 160 (P), Hind Plast Super 16..."
9,0,HC015,Hind Plast Super 160 (P),HANDLING PRECAUTION,"[Hind Plast Super 160 (P), Hind Plast Super 16..."


In [59]:
row = valid_tds[
    valid_tds["Product_Name"].str.contains(
        "Hind Crystel Seal",
        case=False,
        na=False
    )
].iloc[0]

print("Product:", row["Product_Name"])
print("Product ID:", row["Product_ID"])
print("=" * 100)
print(row["TDS_Text"])

Product: Hind Crystel Seal
Product ID: HC001
HINDCON
CHEMICALS LTD.HIND CRYSTAL SEAL
DESCRIPTION 
Hind Crystal Seal is a capillary waterprooﬁng formulation of proprietary blends of moisture active 
chemicals, quartz, sand and cement. The chemical contained in Hind Crystal Seal formulations 
requires the presence of moisture for the chemical reaction to take place and in combination with 
the “free lime”, the chemical in Hind Crystal Seal forms long chain complexes which crystallizes 
in the capillaries. It conforms to IS: 2645-2003 & ACI 212.3R - 10.
USES 
Hind Crystal Seal protects concrete and mortar making it 100% waterproof. It improves the 
adhesion between top coat and concrete or mortar. It is used for waterprooﬁng of water reservoir, 
water tower, tunnels, dams, canals, concrete pipes, water treatment plant, sea wall, bridges and 
decks etc.
ADVANTAGES
• Ready for use only by mixing with water.
• Can be applied on damp concrete surface with no standing water.
• Non-toxic and ca

In [60]:
row = valid_tds[
    valid_tds["Product_Name"].str.contains(
        "Hind Plug",
        case=False,
        na=False
    )
].iloc[0]

print("Product:", row["Product_Name"])
print("Product ID:", row["Product_ID"])
print("=" * 100)
print(row["TDS_Text"])

Product: Hind Plug - S
Product ID: HC003
HINDCONCHEMICALS LTD.
HIND PLUG - S 
DESCRIPTION 
Hind Plug S is fast setting powdered compound for stopping running water. 
USES 
It is used to stop running water and seepage through concrete and masonry walls, dams etc.
ADVANTAGES
• Stops running water immediately
• Non shrink and Non-metallic.
• High strength.
• Easy to use.
• Sets under water.
APPLICATION
Chisel the masonry around the crack to a width & depth not less than 25 mm. The hole should be 
wider within the wall than at the surface. Mix Hind Plug S in the ratio of 0.28 to 0.33 (Water/powder 
ratio). Force the material into the crack when heat generation is felt. Hold tightly with hand for 3 
minutes, wearing rubber hand gloves.
PROPERTIES
  Aspect       Light Brown Powder
o
  Setting at 27 C      1 to 3 minutes
2
  Compressive Strength     8 N/mm  (minimum) after 1 hour
2
         19 N/mm  (minimum) after 24 hour
SHELF LIFE
Best before 6 months from the date of manufacture. Should b

In [61]:
results = vectorstore.similarity_search(
    query,
    k=3
)

In [62]:
results = vectorstore.similarity_search(
    query="What is the coverage?",
    k=5,
    filter={
        "product_id": "HC001"
    }
)

In [63]:
product_lookup = {}

for _, row in valid_tds.iterrows():
    product_name = str(row["Product_Name"]).strip()
    product_id = str(row["Product_ID"]).strip()

    product_lookup[product_name.lower()] = product_id

print("Products:", len(product_lookup))

Products: 244


In [66]:
query_product = "Hind Super FA"

product_id = product_lookup.get(query_product.lower())

print(product_id)

HC069


In [67]:
def detect_product(query: str):
    query_lower = query.lower()

    matches = []

    for product_name, product_id in product_lookup.items():

        if product_name in query_lower:
            matches.append({
                "product_name": product_name,
                "product_id": product_id
            })

    return matches

In [68]:
query = "What is the coverage of Hind Crystel Seal?"

matches = detect_product(query)

print(matches)

[{'product_name': 'hind crystel seal', 'product_id': 'HC001'}]


In [72]:
def search_tds(
    query: str,
    k: int = 5,
    product_id: str | None = None,
    section: str | None = None
):
    conditions = []

    if product_id:
        conditions.append({
            "product_id": product_id
        })

    if section:
        conditions.append({
            "section": section
        })

    # No filters
    if len(conditions) == 0:
        return vectorstore.similarity_search(
            query,
            k=k
        )

    # One filter
    if len(conditions) == 1:
        return vectorstore.similarity_search(
            query,
            k=k,
            filter=conditions[0]
        )

    # Multiple filters
    return vectorstore.similarity_search(
        query,
        k=k,
        filter={
            "$and": conditions
        }
    )

In [73]:
results = search_tds(
    query="What is the coverage?",
    k=3,
    product_id="HC001",
    section="COVERAGE"
)

In [75]:
def display_results(results):
    for rank, doc in enumerate(results, start=1):
        print("=" * 90)
        print(f"RESULT {rank}")
        print("Product:", doc.metadata.get("product_name"))
        print("Product ID:", doc.metadata.get("product_id"))
        print("Section:", doc.metadata.get("section"))
        print("Document:", doc.metadata.get("document_name"))
        print("-" * 90)
        print(doc.page_content)

In [76]:
display_results(results)

RESULT 1
Product: Hind Crystel Seal
Product ID: HC001
Section: COVERAGE
Document: HIND-CRYSTAL-SEAL_TDS-1.pdf
------------------------------------------------------------------------------------------
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


In [77]:
results = search_tds(
    query="What is the coverage?",
    k=3,
    product_id="HC001",
    section="COVERAGE"
)

display_results(results)

RESULT 1
Product: Hind Crystel Seal
Product ID: HC001
Section: COVERAGE
Document: HIND-CRYSTAL-SEAL_TDS-1.pdf
------------------------------------------------------------------------------------------
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


In [87]:
for doc in final_chunks:
    if doc.metadata.get("product_id") == "HC001":
        print("=" * 100)
        print("PRODUCT:", doc.metadata.get("product_name"))
        print("PRODUCT ID:", doc.metadata.get("product_id"))
        print("SECTION:", doc.metadata.get("section"))
        print("-" * 100)
        print(doc.page_content[:2000])

PRODUCT: Hind Crystel Seal
PRODUCT ID: HC001
SECTION: GENERAL
----------------------------------------------------------------------------------------------------
HINDCON
CHEMICALS LTD.HIND CRYSTAL SEAL
PRODUCT: Hind Crystel Seal
PRODUCT ID: HC001
SECTION: DESCRIPTION
----------------------------------------------------------------------------------------------------
Hind Crystal Seal is a capillary waterprooﬁng formulation of proprietary blends of moisture active
chemicals, quartz, sand and cement. The chemical contained in Hind Crystal Seal formulations
requires the presence of moisture for the chemical reaction to take place and in combination with
the “free lime”, the chemical in Hind Crystal Seal forms long chain complexes which crystallizes
in the capillaries. It conforms to IS: 2645-2003 & ACI 212.3R - 10.
PRODUCT: Hind Crystel Seal
PRODUCT ID: HC001
SECTION: USES
----------------------------------------------------------------------------------------------------
Hind Crystal Se

In [88]:
coverage_chunks = [
    doc for doc in final_chunks
    if doc.metadata.get("product_id") == "HC001"
    and doc.metadata.get("section") == "COVERAGE"
]

print("HC001 COVERAGE chunks:", len(coverage_chunks))

for doc in coverage_chunks:
    print("=" * 100)
    print(doc.page_content)

HC001 COVERAGE chunks: 1
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


In [89]:
sections = sorted(
    set(
        doc.metadata.get("section")
        for doc in final_chunks
        if doc.metadata.get("product_id") == "HC001"
    )
)

print(sections)

['ADVANTAGES', 'APPLICATION', 'COVERAGE', 'DESCRIPTION', 'GENERAL', 'HANDLING PRECAUTION', 'PACKING', 'PROPERTIES', 'SHELF LIFE', 'USES']


In [90]:
results = search_tds(
    query="What is the coverage?",
    k=5,
    product_id="HC001"
)

for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print("RESULT:", i)
    print("Product:", doc.metadata.get("product_name"))
    print("Product ID:", doc.metadata.get("product_id"))
    print("Section:", doc.metadata.get("section"))
    print(doc.page_content[:500])

RESULT: 1
Product: Hind Crystel Seal
Product ID: HC001
Section: PROPERTIES
Aspect Grey Powder
Bulk density gm/cc 1.25
Water Penetration (BS EN 12390 Part 8:2000) Nil
Water Pressure Head (mtr) 40-50
Corrosion Resistance (pH) (mixed with water) 10 + 1
Particle Size (micron) 40-150
Heat Resistance (ASTM C 1064-86) Same as Concrete
Service Life Same as RCC
RESULT: 2
Product: Hind Crystel Seal
Product ID: HC001
Section: APPLICATION
the safety and fitness for the product for use.
The right to change the properties of the products is reserved with us. The proprietary rights of third parties must be observed. All orders are accepted subject to the terms of sale and delivery. Users must
always refer to the most recent issue of the latest Data Sheet for the product concerned, copies of which will be supplied on request.
RESULT: 3
Product: Hind Crystel Seal
Product ID: HC001
Section: APPLICATION
Hindcon Chemicals maintains a team of technically trained professionals to provide full support to you

In [91]:
results = search_tds(
    query="What is the coverage?",
    k=3,
    product_id="HC001",
    section="COVERAGE"
)

for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print("RESULT:", i)
    print("Product:", doc.metadata.get("product_name"))
    print("Product ID:", doc.metadata.get("product_id"))
    print("Section:", doc.metadata.get("section"))
    print(doc.page_content[:1000])

RESULT: 1
Product: Hind Crystel Seal
Product ID: HC001
Section: COVERAGE
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


Phase 4


In [92]:
# Create product master from the dataset

product_master = (
    valid_tds[
        ["Product_ID", "Product_Name", "Category"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Total unique products:", len(product_master))
product_master.head(20)

Total unique products: 258


,Product_ID,Product_Name,Category
0,HC001,Hind Crystel Seal,Waterproofing Compounds
1,HC002,Hind Crystel Seal (1),Waterproofing Compounds
2,HC003,Hind Plug - S,Waterproofing Compounds
3,HC004,Hind Proof No – 1,Waterproofing Compounds
4,HC005,Hind Proof No – 2,Waterproofing Compounds
5,HC006,Hind Proof No – 3,Waterproofing Compounds
6,HC007,Hind Powder WP,Waterproofing Compounds
7,HC008,Hind Flow Set 1000,Concrete & Mortar Admixtures
8,HC009,Hind Plast AEA,Concrete & Mortar Admixtures
9,HC010,Hind Plast AEP,Concrete & Mortar Admixtures


In [93]:
import re

def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = re.sub(r"[^a-z0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [94]:
test_names = [
    "Hind Crystel Seal",
    "HIND CRYSTEL SEAL",
    "Hind Crystal Seal",
    "hind crystel seal"
]

for name in test_names:
    print(name, "→", normalize_text(name))

Hind Crystel Seal → hind crystel seal
HIND CRYSTEL SEAL → hind crystel seal
Hind Crystal Seal → hind crystal seal
hind crystel seal → hind crystel seal


In [95]:
product_lookup = {}

for _, row in product_master.iterrows():

    product_id = str(row["Product_ID"]).strip()
    product_name = str(row["Product_Name"]).strip()

    normalized_name = normalize_text(product_name)

    product_lookup[normalized_name] = {
        "product_id": product_id,
        "product_name": product_name,
        "category": str(row["Category"]).strip()
    }

print("Products in lookup:", len(product_lookup))

Products in lookup: 244


In [96]:
product_lookup.get(normalize_text("Hind Crystel Seal"))

{'product_id': 'HC001',
 'product_name': 'Hind Crystel Seal',
 'category': 'Waterproofing Compounds'}

In [306]:
def detect_product(query: str):
    normalized_query = normalize_product_query(query)
    matches = []

    for normalized_name, product_info in product_lookup.items():
        if normalized_name in normalized_query:
            matches.append((len(normalized_name), product_info))

    # No product found
    if not matches:
        return {
            "status": "NOT_FOUND",
            "products": []
        }

    # Sort by product-name length, longest first
    matches.sort(key=lambda x: x[0], reverse=True)

    # Take only the longest matching product name
    longest_length = matches[0][0]

    best_matches = [
        product_info
        for length, product_info in matches
        if length == longest_length
    ]

    # Remove duplicate product IDs
    unique_products = {}

    for product_info in best_matches:
        unique_products[product_info["product_id"]] = product_info

    best_matches = list(unique_products.values())

    # Exactly one best product
    if len(best_matches) == 1:
        return {
            "status": "FOUND",
            "product": best_matches[0]
        }

    # Multiple products have exactly the same best match
    return {
        "status": "AMBIGUOUS",
        "products": best_matches
    }

In [98]:
queries = [
    "What is the coverage of Hind Crystel Seal?",
    "Tell me the shelf life of Hind Crystel Seal",
    "What is the packing of Hind Plug - S?",
    "What is the water pressure head?"
]

for query in queries:

    result = detect_product(query)

    print("=" * 80)
    print("QUERY:", query)
    print("RESULT:", result)

QUERY: What is the coverage of Hind Crystel Seal?
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
QUERY: Tell me the shelf life of Hind Crystel Seal
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
QUERY: What is the packing of Hind Plug - S?
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC003', 'product_name': 'Hind Plug - S', 'category': 'Waterproofing Compounds'}}
QUERY: What is the water pressure head?
RESULT: {'status': 'NOT_FOUND', 'products': []}


In [111]:
SECTION_ALIASES = {
    "COVERAGE": [
        "coverage",
        "consumption",
        "quantity required",
        "material required",
        "how much material",
        "application rate"
    ],

    "SHELF LIFE": [
        "shelf life",
        "storage life",
        "expiry",
        "expiration"
    ],

    "PACKING": [
        "packing",
        "package",
        "pack size",
        "packaging"
    ],

    "DESCRIPTION": [
    "description",
    "what does it do",
    "describe the product"
    ],

    "USES": [
        "uses",
        "used for",
        "where is it used",
        "purpose"
    ],

    "ADVANTAGES": [
        "advantages",
        "benefits",
        "features"
    ],

    "APPLICATION": [
        "application",
        "how to apply",
        "application method",
        "procedure",
        "method of application"
    ],

    "PROPERTIES": [
        "properties",
        "physical properties",
        "technical properties"
    ],

    "TECHNICAL DATA": [
        "technical data",
        "technical information"
    ],

    "SPECIFICATIONS": [
        "specifications",
        "specification"
    ],

    "LIMITATIONS": [
        "limitations",
        "limitations of use",
        "restrictions"
    ]
}

In [205]:
def detect_section(query: str):
    normalized_query = normalize_text(query)

    matches = []

    for section, aliases in SECTION_ALIASES.items():

        for alias in aliases:

            normalized_alias = normalize_text(alias)

            if normalized_alias in normalized_query:
                matches.append(section)
                break

    # Remove duplicates while preserving order
    matches = list(dict.fromkeys(matches))

    if len(matches) == 1:
        return {
            "status": "FOUND",
            "section": matches[0]
        }

    if len(matches) > 1:
        return {
            "status": "MULTIPLE",
            "sections": matches
        }

    return {
        "status": "NOT_FOUND",
        "sections": []
    }

In [206]:
section_queries = [
    "What is the coverage?",
    "What is the shelf life?",
    "How is it packed?",
    "What are the advantages?",
    "How do I apply it?",
    "What are the properties?"
]

for query in section_queries:

    result = detect_section(query)

    print("=" * 80)
    print("QUERY:", query)
    print("RESULT:", result)

QUERY: What is the coverage?
RESULT: {'status': 'FOUND', 'section': 'COVERAGE'}
QUERY: What is the shelf life?
RESULT: {'status': 'FOUND', 'section': 'SHELF LIFE'}
QUERY: How is it packed?
RESULT: {'status': 'NOT_FOUND', 'sections': []}
QUERY: What are the advantages?
RESULT: {'status': 'FOUND', 'section': 'ADVANTAGES'}
QUERY: How do I apply it?
RESULT: {'status': 'NOT_FOUND', 'sections': []}
QUERY: What are the properties?
RESULT: {'status': 'FOUND', 'section': 'PROPERTIES'}


In [207]:
def analyze_query(query: str):

    product_result = detect_product(query)
    section_result = detect_section(query)

    return {
        "query": query,
        "product_detection": product_result,
        "section_detection": section_result
    }

In [208]:
query = "What is the coverage of Hind Crystel Seal?"

analysis = analyze_query(query)

analysis

{'query': 'What is the coverage of Hind Crystel Seal?',
 'product_detection': {'status': 'FOUND',
  'product': {'product_id': 'HC001',
   'product_name': 'Hind Crystel Seal',
   'category': 'Waterproofing Compounds'}},
 'section_detection': {'status': 'FOUND', 'section': 'COVERAGE'}}

In [209]:
def intelligent_search(query: str, k: int = 5):

    analysis = analyze_query(query)

    product_result = analysis["product_detection"]
    section_result = analysis["section_detection"]

    product_id = None
    section = None

    # Product detected
    if product_result["status"] == "FOUND":
        product_id = product_result["product"]["product_id"]

    # Section detected
    if section_result["status"] == "FOUND":
        section = section_result["section"]

    results = search_tds(
        query=query,
        k=k,
        product_id=product_id,
        section=section
    )

    return results, analysis

In [210]:
query = "What is the shelf life of Hind Crystel Seal?"

results, analysis = intelligent_search(query)

print("QUERY:", query)
print()
print("PRODUCT DETECTION:")
print(analysis["product_detection"])

print()
print("SECTION DETECTION:")
print(analysis["section_detection"])

print()
print("RETRIEVED DOCUMENTS")
print("=" * 100)

for i, doc in enumerate(results, start=1):

    print("RESULT:", i)
    print("Product:", doc.metadata.get("product_name"))
    print("Product ID:", doc.metadata.get("product_id"))
    print("Section:", doc.metadata.get("section"))
    print("-" * 100)
    print(doc.page_content[:1000])
    print()

QUERY: What is the shelf life of Hind Crystel Seal?

PRODUCT DETECTION:
{'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

SECTION DETECTION:
{'status': 'FOUND', 'section': 'SHELF LIFE'}

RETRIEVED DOCUMENTS
RESULT: 1
Product: Hind Crystel Seal
Product ID: HC001
Section: SHELF LIFE
----------------------------------------------------------------------------------------------------
Best before 12 months from the date of manufacture. Should be stored in a cool and dry place
o o
away from direct sunlight at temperature ranging between 2 C (Min.) and 45 C (Max.).



Phase - 5


In [211]:
product_master = (
    valid_tds[
        ["Product_ID", "Product_Name", "Category"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Total unique products:", len(product_master))

Total unique products: 258


In [212]:
product_aliases = {}

for _, row in product_master.iterrows():

    product_id = str(row["Product_ID"]).strip()
    product_name = str(row["Product_Name"]).strip()
    category = str(row["Category"]).strip()

    normalized_name = normalize_text(product_name)

    product_aliases[normalized_name] = {
        "product_id": product_id,
        "product_name": product_name,
        "category": category
    }

print("Total product aliases:", len(product_aliases))

Total product aliases: 244


In [213]:
def normalize_product_query(text: str) -> str:

    text = normalize_text(text)

    # Common spelling corrections
    replacements = {
        "crystal": "crystel",
    }

    words = text.split()

    corrected_words = []

    for word in words:
        corrected_words.append(
            replacements.get(word, word)
        )

    return " ".join(corrected_words)

In [214]:
print(normalize_product_query("Hind Crystal Seal"))

hind crystel seal


In [215]:
print(normalize_product_query("Hind Crystel Seal"))

hind crystel seal


In [216]:
test_queries = [
    "What is the coverage of Hind Crystel Seal?",
    "What is the coverage of Hind Crystal Seal?",
    "Tell me the shelf life of Hind Crystel Seal",
    "Hind Crystel Seal packing",
    "What is the coverage?"
]

for query in test_queries:

    result = detect_product(query)

    print("\nQUERY:", query)
    print("RESULT:", result)


QUERY: What is the coverage of Hind Crystel Seal?
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

QUERY: What is the coverage of Hind Crystal Seal?
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

QUERY: Tell me the shelf life of Hind Crystel Seal
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

QUERY: Hind Crystel Seal packing
RESULT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

QUERY: What is the coverage?
RESULT: {'status': 'NOT_FOUND', 'products': []}


In [217]:
SECTION_ALIASES = {

    "COVERAGE": [
        "coverage",
        "consumption",
        "quantity required",
        "material required",
        "how much material",
        "application rate"
    ],

    "SHELF LIFE": [
        "shelf life",
        "storage life",
        "expiry",
        "expiration",
        "validity"
    ],

    "PACKING": [
        "packing",
        "package",
        "pack size",
        "packaging"
    ],

    "DESCRIPTION": [
        "description",
        "what does it do",
        "describe the product"
    ],

    "USES": [
        "uses",
        "used for",
        "where is it used",
        "purpose"
    ],

    "ADVANTAGES": [
        "advantages",
        "benefits",
        "features"
    ],

    "APPLICATION": [
        "application",
        "how to apply",
        "application method",
        "procedure",
        "method of application"
    ],

    "PROPERTIES": [
        "properties",
        "physical properties",
        "technical properties"
    ],

    "TECHNICAL DATA": [
        "technical data",
        "technical information"
    ],

    "SPECIFICATIONS": [
        "specifications",
        "specification"
    ],

    "LIMITATIONS": [
        "limitations",
        "limitations of use",
        "restrictions"
    ]
}

In [218]:
queries = [
    "What is the coverage of Hind Crystel Seal?",
    "What is the shelf life of Hind Crystel Seal?",
    "What is the packing of Hind Crystal Seal?",
    "What are the advantages of Hind Crystel Seal?",
    "How do I apply Hind Crystel Seal?",
    "What is the coverage?"
]

for query in queries:

    product_result = detect_product(query)
    section_result = detect_section(query)

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("PRODUCT:", product_result)
    print("SECTION:", section_result)


QUERY: What is the coverage of Hind Crystel Seal?
PRODUCT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
SECTION: {'status': 'FOUND', 'section': 'COVERAGE'}

QUERY: What is the shelf life of Hind Crystel Seal?
PRODUCT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
SECTION: {'status': 'FOUND', 'section': 'SHELF LIFE'}

QUERY: What is the packing of Hind Crystal Seal?
PRODUCT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
SECTION: {'status': 'FOUND', 'section': 'PACKING'}

QUERY: What are the advantages of Hind Crystel Seal?
PRODUCT: {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
SECTION: {'status': 'FOUND', 'section': 'ADVANTAGES'}

QUERY: How do I 

In [219]:
def route_query(query: str):

    product_result = detect_product(query)
    section_result = detect_section(query)

    # -----------------------------------
    # Product not specified
    # -----------------------------------

    if product_result["status"] == "NOT_FOUND":

        return {
            "route": "CLARIFICATION_REQUIRED",
            "reason": "PRODUCT_NOT_SPECIFIED",
            "message": "Please specify the product name."
        }

    # -----------------------------------
    # Multiple products detected
    # -----------------------------------

    if product_result["status"] == "AMBIGUOUS":

        return {
            "route": "CLARIFICATION_REQUIRED",
            "reason": "MULTIPLE_PRODUCTS",
            "products": product_result["products"],
            "message": "Please specify which product you mean."
        }

    # -----------------------------------
    # Product found
    # -----------------------------------

    product = product_result["product"]

    # -----------------------------------
    # Multiple sections detected
    # -----------------------------------

    if section_result["status"] == "MULTIPLE":

        return {
            "route": "RETRIEVE_WITH_PRODUCT",
            "product": product,
            "reason": "MULTIPLE_SECTIONS"
        }

    # -----------------------------------
    # Section found
    # -----------------------------------

    if section_result["status"] == "FOUND":

        return {
            "route": "RETRIEVE_WITH_PRODUCT_AND_SECTION",
            "product": product,
            "section": section_result["section"]
        }

    # -----------------------------------
    # Product found but section not found
    # -----------------------------------

    return {
        "route": "RETRIEVE_WITH_PRODUCT",
        "product": product,
        "reason": "SECTION_NOT_SPECIFIED"
    }

In [220]:
test_queries = [
    "What is the coverage of Hind Crystel Seal?",
    "What is the shelf life of Hind Crystel Seal?",
    "What is the coverage?",
    "Tell me about Hind Crystel Seal",
    "What are the advantages of Hind Crystal Seal?"
]

for query in test_queries:

    result = route_query(query)

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("ROUTE:", result)


QUERY: What is the coverage of Hind Crystel Seal?
ROUTE: {'route': 'RETRIEVE_WITH_PRODUCT_AND_SECTION', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}, 'section': 'COVERAGE'}

QUERY: What is the shelf life of Hind Crystel Seal?
ROUTE: {'route': 'RETRIEVE_WITH_PRODUCT_AND_SECTION', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}, 'section': 'SHELF LIFE'}

QUERY: What is the coverage?
ROUTE: {'route': 'CLARIFICATION_REQUIRED', 'reason': 'PRODUCT_NOT_SPECIFIED', 'message': 'Please specify the product name.'}

QUERY: Tell me about Hind Crystel Seal
ROUTE: {'route': 'RETRIEVE_WITH_PRODUCT', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}, 'reason': 'SECTION_NOT_SPECIFIED'}

QUERY: What are the advantages of Hind Crystal Seal?
ROUTE: {'route': 'RETRIEVE_WITH_PRODUCT_AND_SECTION', 'product': {'product_id'

In [221]:
def intelligent_search(
    query: str,
    k: int = 5
):

    routing = route_query(query)

    route = routing["route"]

    # --------------------------------
    # Clarification
    # --------------------------------

    if route == "CLARIFICATION_REQUIRED":

        return [], routing

    # --------------------------------
    # Product + Section
    # --------------------------------

    if route == "RETRIEVE_WITH_PRODUCT_AND_SECTION":

        product_id = routing[
            "product"
        ]["product_id"]

        section = routing["section"]

        results = hybrid_search(
            query=query,
            k=k,
            product_id=product_id,
            section=section
        )

        return results, routing

    # --------------------------------
    # Product only
    # --------------------------------

    if route == "RETRIEVE_WITH_PRODUCT":

        product_id = routing[
            "product"
        ]["product_id"]

        results = hybrid_search(
            query=query,
            k=k,
            product_id=product_id
        )

        return results, routing

    return [], routing

In [222]:
queries = [
    "What is the coverage of Hind Crystel Seal?",
    "What is the shelf life of Hind Crystel Seal?",
    "What is the coverage?",
    "Tell me about Hind Crystel Seal",
    "What is the packing of Hind Crystal Seal?"
]

for query in queries:

    results, routing = intelligent_search(query)

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("ROUTING:", routing)

    if routing["route"] == "CLARIFICATION_REQUIRED":

        print("SYSTEM:", routing["message"])

    else:

        print("RETRIEVED DOCUMENTS:")

        for i, doc in enumerate(results, 1):

            print(
                f"{i}. "
                f"Product={doc.metadata.get('product_name')} | "
                f"Section={doc.metadata.get('section')}"
            )


QUERY: What is the coverage of Hind Crystel Seal?
ROUTING: {'route': 'RETRIEVE_WITH_PRODUCT_AND_SECTION', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}, 'section': 'COVERAGE'}
RETRIEVED DOCUMENTS:
1. Product=Hind Crystel Seal | Section=COVERAGE

QUERY: What is the shelf life of Hind Crystel Seal?
ROUTING: {'route': 'RETRIEVE_WITH_PRODUCT_AND_SECTION', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}, 'section': 'SHELF LIFE'}
RETRIEVED DOCUMENTS:
1. Product=Hind Crystel Seal | Section=SHELF LIFE

QUERY: What is the coverage?
ROUTING: {'route': 'CLARIFICATION_REQUIRED', 'reason': 'PRODUCT_NOT_SPECIFIED', 'message': 'Please specify the product name.'}
SYSTEM: Please specify the product name.

QUERY: Tell me about Hind Crystel Seal
ROUTING: {'route': 'RETRIEVE_WITH_PRODUCT', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Water

Phase - 6

In [223]:
def keyword_search(
    query: str,
    documents: list,
    k: int = 5,
    product_id: str | None = None,
    section: str | None = None
):
    
    query_normalized = normalize_text(query)
    
    # Remove common question words
    stop_words = {
        "what",
        "is",
        "the",
        "of",
        "for",
        "how",
        "does",
        "do",
        "tell",
        "me",
        "about",
        "please",
        "can",
        "you",
        "give",
        "which"
    }
    
    query_terms = [
        word
        for word in query_normalized.split()
        if word not in stop_words
    ]
    
    scored_documents = []
    
    for doc in documents:
        
        # -----------------------------
        # Metadata filtering
        # -----------------------------
        
        if product_id:
            if doc.metadata.get("product_id") != product_id:
                continue
        
        if section:
            if doc.metadata.get("section") != section:
                continue
        
        text = normalize_text(doc.page_content)
        
        if not text:
            continue
        
        score = 0
        
        # -----------------------------
        # Keyword matching
        # -----------------------------
        
        for term in query_terms:
            
            if term in text:
                score += 1
        
        # -----------------------------
        # Exact phrase matching
        # -----------------------------
        
        query_without_product = query_normalized
        
        if query_without_product in text:
            score += 5
        
        if score > 0:
            scored_documents.append(
                (score, doc)
            )
    
    # Highest score first
    scored_documents.sort(
        key=lambda x: x[0],
        reverse=True
    )
    
    return [
        doc
        for score, doc in scored_documents[:k]
    ]

In [224]:
results = keyword_search(
    query="What is the coverage of Hind Crystel Seal?",
    documents=final_chunks,
    k=5,
    product_id="HC001",
    section="COVERAGE"
)

for i, doc in enumerate(results, 1):
    print(
        f"{i}. "
        f"{doc.metadata['product_name']} | "
        f"{doc.metadata['section']}"
    )
    print(doc.page_content[:500])
    print()

In [225]:
def extract_query_terms(query: str):

    normalized_query = normalize_text(query)

    stop_words = {
        "what",
        "is",
        "the",
        "of",
        "for",
        "how",
        "does",
        "do",
        "tell",
        "me",
        "about",
        "please",
        "can",
        "you",
        "give",
        "which",
        "tell"
    }

    words = [
        word
        for word in normalized_query.split()
        if word not in stop_words
    ]

    terms = words.copy()

    # Add bigrams
    for i in range(len(words) - 1):
        terms.append(
            f"{words[i]} {words[i+1]}"
        )

    return terms

In [226]:
print(
    extract_query_terms(
        "What is the compressive strength of Hind Plug - S?"
    )
)

['compressive', 'strength', 'hind', 'plug', '-', 's', 'compressive strength', 'strength hind', 'hind plug', 'plug -', '- s']


In [227]:
def keyword_search(
    query: str,
    documents: list,
    k: int = 5,
    product_id: str | None = None,
    section: str | None = None
):

    query_terms = extract_query_terms(query)

    scored_documents = []

    for doc in documents:

        # -----------------------------
        # Product filter
        # -----------------------------

        if product_id:
            if doc.metadata.get("product_id") != product_id:
                continue

        # -----------------------------
        # Section filter
        # -----------------------------

        if section:
            if doc.metadata.get("section") != section:
                continue

        text = normalize_text(
            doc.page_content
        )

        if not text:
            continue

        score = 0

        # -----------------------------
        # Score each query term
        # -----------------------------

        for term in query_terms:

            if " " in term:
                
                # Multi-word phrase
                if term in text:
                    score += 3

            else:

                if term in text:
                    score += 1

        # -----------------------------
        # Exact phrase
        # -----------------------------

        normalized_query = normalize_text(query)

        if normalized_query in text:
            score += 5

        if score > 0:

            scored_documents.append(
                (score, doc)
            )

    scored_documents.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in scored_documents[:k]
    ]

In [228]:
def hybrid_search(
    query: str,
    k: int = 5,
    product_id: str | None = None,
    section: str | None = None
):

    # --------------------------------
    # Semantic results
    # --------------------------------

    semantic_results = search_tds(
        query=query,
        k=k,
        product_id=product_id,
        section=section
    )

    # --------------------------------
    # Keyword results
    # --------------------------------

    keyword_results = keyword_search(
        query=query,
        documents=final_chunks,
        k=k,
        product_id=product_id,
        section=section
    )

    # --------------------------------
    # Combine results
    # --------------------------------

    combined = []

    seen = set()

    for doc in keyword_results + semantic_results:

        chunk_id = doc.metadata.get(
            "chunk_id"
        )

        if chunk_id not in seen:

            combined.append(doc)
            seen.add(chunk_id)

    return combined[:k]

In [229]:
query = "What is the coverage of Hind Crystel Seal?"

results = hybrid_search(
    query=query,
    k=5,
    product_id="HC001",
    section="COVERAGE"
)

for i, doc in enumerate(results, 1):

    print("=" * 80)

    print(
        f"RESULT {i}"
    )

    print(
        "Product:",
        doc.metadata.get("product_name")
    )

    print(
        "Section:",
        doc.metadata.get("section")
    )

    print(
        doc.page_content[:700]
    )

RESULT 1
Product: Hind Crystel Seal
Section: COVERAGE
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


In [230]:
test_queries = [

    "What is the coverage of Hind Crystel Seal?",

    "What is the shelf life of Hind Crystel Seal?",

    "What is the packing of Hind Crystel Seal?",

    "What is the water pressure head of Hind Crystel Seal?",

    "What is the particle size of Hind Crystel Seal?",

    "What is the compressive strength of Hind Plug - S?",

    "What is the setting time of Hind Plug - S?",

]

In [231]:
for query in test_queries:

    print("\n")
    print("=" * 100)
    print("QUERY:", query)

    results, routing = intelligent_search(
        query,
        k=3
    )

    print(
        "ROUTE:",
        routing["route"]
    )

    for i, doc in enumerate(results, 1):

        print(
            f"\nRESULT {i}"
        )

        print(
            "Product:",
            doc.metadata.get("product_name")
        )

        print(
            "Section:",
            doc.metadata.get("section")
        )

        print(
            doc.page_content[:400]
        )



QUERY: What is the coverage of Hind Crystel Seal?
ROUTE: RETRIEVE_WITH_PRODUCT_AND_SECTION

RESULT 1
Product: Hind Crystel Seal
Section: COVERAGE
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


QUERY: What is the shelf life of Hind Crystel Seal?
ROUTE: RETRIEVE_WITH_PRODUCT_AND_SECTION

RESULT 1
Product: Hind Crystel Seal
Section: SHELF LIFE
Best before 12 months from the date of manufacture. Should be stored in a cool and dry place
o o
away from direct sunlight at temperature ranging between 2 C (Min.) and 45 C (Max.).


QUERY: What is the packing of Hind Crystel Seal?
ROUTE: RETRIEVE_WITH_PRODUCT_AND_SECTION

RESULT 1
Product: Hind Crystel Seal
Section: PACKING
10 Kg. in HDPE Bag.


QUERY: What is the water pressure head of Hind Crystel Seal?
ROUTE: RETRIEVE_WITH_PRODUCT

RESULT 1
Product: Hind Crystel Seal
Section: PROPERTIES
Aspect Grey Powder
Bulk density gm/cc 1.25
Water Penetration (BS EN 12390 Part 8:2000) Nil
Water Pressure Head (mt

In [232]:
def remove_duplicate_chunks(documents):

    unique_documents = []
    seen_content = set()

    for doc in documents:

        normalized_content = normalize_text(
            doc.page_content
        )

        if normalized_content in seen_content:
            continue

        seen_content.add(
            normalized_content
        )

        unique_documents.append(doc)

    return unique_documents

In [233]:
def evidence_score(
    query: str,
    document
):

    query_terms = extract_query_terms(query)

    text = normalize_text(
        document.page_content
    )

    score = 0

    for term in query_terms:

        if term in text:

            if " " in term:
                score += 3
            else:
                score += 1

    # Technical section bonus
    technical_sections = {
        "PROPERTIES",
        "TECHNICAL DATA",
        "SPECIFICATIONS",
        "COVERAGE",
        "SHELF LIFE",
        "PACKING"
    }

    if document.metadata.get(
        "section"
    ) in technical_sections:

        score += 2

    return score

In [234]:
def rank_evidence(query, documents):

    scored = []

    for doc in documents:

        score = evidence_score(
            query,
            doc
        )

        scored.append(
            (score, doc)
        )

    scored.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in scored
    ]

In [235]:
def hybrid_search(
    query: str,
    k: int = 5,
    product_id: str | None = None,
    section: str | None = None
):

    semantic_results = search_tds(
        query=query,
        k=k,
        product_id=product_id,
        section=section
    )

    keyword_results = keyword_search(
        query=query,
        documents=final_chunks,
        k=k,
        product_id=product_id,
        section=section
    )

    combined = remove_duplicate_chunks(
        keyword_results + semantic_results
    )

    ranked = rank_evidence(
        query,
        combined
    )

    return ranked[:k]

Phase 7

In [236]:
QUERY_STATUS = {
    "VALID": "VALID",
    "INCOMPLETE_QUERY": "INCOMPLETE_QUERY",
    "PRODUCT_NOT_FOUND": "PRODUCT_NOT_FOUND",
    "INFORMATION_NOT_AVAILABLE": "INFORMATION_NOT_AVAILABLE",
    "INVALID_QUERY": "INVALID_QUERY",
}

In [237]:
def is_obviously_invalid_query(query: str):

    if not isinstance(query, str):
        return True

    query = query.strip()

    if not query:
        return True

    # Very short meaningless inputs
    if len(query) < 3:
        return True

    return False

In [238]:
known_product_words = set()

for _, row in product_master.iterrows():

    product_name = normalize_product_query(
        str(row["Product_Name"])
    )

    for word in product_name.split():

        if len(word) >= 3:

            known_product_words.add(word)

print(
    "Known product words:",
    len(known_product_words)
)

Known product words: 197


In [239]:
def query_contains_product_like_phrase(query: str):

    normalized_query = normalize_product_query(
        query
    )

    words = normalized_query.split()

    for word in words:

        if word in known_product_words:

            return True

    return False

In [240]:
print(
    query_contains_product_like_phrase(
        "What is the coverage of Hind Crystel Seal?"
    )
)

True


In [241]:
print(
    query_contains_product_like_phrase(
        "What is the coverage?"
    )
)

False


In [242]:
def get_product_status(query: str):

    product_result = detect_product(query)

    # Product successfully identified
    if product_result["status"] == "FOUND":

        return {
            "status": "FOUND",
            "product": product_result["product"]
        }

    # Multiple products detected
    if product_result["status"] == "AMBIGUOUS":

        return {
            "status": "AMBIGUOUS",
            "products": product_result["products"]
        }

    # Product not detected
    if query_contains_product_like_phrase(query):

        return {
            "status": "PRODUCT_NOT_FOUND",
            "products": []
        }

    return {
        "status": "NOT_SPECIFIED",
        "products": []
    }

In [243]:
test_queries = [
    "What is the coverage?",
    "What is the coverage of Hind Crystel Seal?",
    "What is the coverage of Hind Crystal Seal?",
    "What is the coverage of ABC Chemical XYZ?"
]

for query in test_queries:

    print("\nQUERY:", query)
    print(
        get_product_status(query)
    )


QUERY: What is the coverage?
{'status': 'NOT_SPECIFIED', 'products': []}

QUERY: What is the coverage of Hind Crystel Seal?
{'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

QUERY: What is the coverage of Hind Crystal Seal?
{'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}

QUERY: What is the coverage of ABC Chemical XYZ?
{'status': 'NOT_SPECIFIED', 'products': []}


In [244]:
def check_evidence(query: str, documents: list):
    
    if not documents:
        return {
            "sufficient": False,
            "score": 0,
            "reason": "NO_DOCUMENTS"
        }

    ranked_documents = rank_evidence(query, documents)

    best_document = ranked_documents[0]

    score = evidence_score(
        query,
        best_document
    )

    section = best_document.metadata.get("section")

    # Strong evidence if the document belongs
    # to the section detected from the question
    analysis = analyze_query(query)

    detected_section = analysis["section_detection"]

    if (
        detected_section["status"] == "FOUND"
        and section == detected_section["section"]
    ):
        score += 5

    return {
        "sufficient": score >= 3,
        "score": score,
        "reason": (
            "SUFFICIENT"
            if score >= 3
            else "WEAK_EVIDENCE"
        ),
        "best_document": best_document
    }

In [245]:
def validate_query(
    query: str
):

    # --------------------------------
    # 1. Basic validation
    # --------------------------------

    if is_obviously_invalid_query(query):

        return {
            "status": QUERY_STATUS["INVALID_QUERY"],
            "message": "Please enter a valid question."
        }

    # --------------------------------
    # 2. Product validation
    # --------------------------------

    product_status = get_product_status(
        query
    )

    # Product missing
    if product_status["status"] == "NOT_SPECIFIED":

        return {
            "status": QUERY_STATUS["INCOMPLETE_QUERY"],
            "message": (
                "Please specify the product "
                "name."
            )
        }

    # Product doesn't exist
    if product_status["status"] == "PRODUCT_NOT_FOUND":

        return {
            "status": QUERY_STATUS["PRODUCT_NOT_FOUND"],
            "message": (
                "The specified product "
                "was not found in the TDS database."
            )
        }

    # Multiple products
    if product_status["status"] == "AMBIGUOUS":

        return {
            "status": QUERY_STATUS["INCOMPLETE_QUERY"],
            "message": (
                "Please specify which product "
                "you mean."
            ),
            "products": product_status[
                "products"
            ]
        }

    # --------------------------------
    # Product found
    # --------------------------------

    return {
        "status": QUERY_STATUS["VALID"],
        "product": product_status["product"]
    }

In [246]:
def retrieve_with_validation(
    query: str,
    k: int = 5
):

    # --------------------------------
    # Query validation
    # --------------------------------

    validation = validate_query(
        query
    )

    if validation["status"] != "VALID":

        return {
            "status": validation["status"],
            "message": validation["message"],
            "documents": [],
            "validation": validation
        }

    # --------------------------------
    # Intelligent retrieval
    # --------------------------------

    documents, routing = intelligent_search(
        query,
        k=k
    )

    # --------------------------------
    # Evidence check
    # --------------------------------

    evidence = check_evidence(
        query,
        documents
    )

    # --------------------------------
    # Evidence insufficient
    # --------------------------------

    if not evidence["sufficient"]:

        return {
            "status": QUERY_STATUS[
                "INFORMATION_NOT_AVAILABLE"
            ],
            "message": (
                "The requested information "
                "could not be reliably found "
                "in the available TDS."
            ),
            "documents": documents,
            "validation": validation,
            "routing": routing,
            "evidence": evidence
        }

    # --------------------------------
    # Valid + sufficient evidence
    # --------------------------------

    return {
        "status": QUERY_STATUS["VALID"],
        "message": "Relevant TDS information found.",
        "documents": documents,
        "validation": validation,
        "routing": routing,
        "evidence": evidence
    }

In [247]:
test_queries = [

    "What is the coverage of Hind Crystel Seal?",

    "What is the shelf life of Hind Crystel Seal?",

    "What is the packing of Hind Crystal Seal?",

    "What is the coverage?",

    "What is the coverage of ABC Chemical XYZ?",

    "Who is the Prime Minister of India?",

    "",

]

In [248]:
for query in test_queries:

    print("\n")
    print("=" * 100)

    print("QUERY:", query)

    result = retrieve_with_validation(
        query,
        k=5
    )

    print(
        "STATUS:",
        result["status"]
    )

    print(
        "MESSAGE:",
        result["message"]
    )

    if result["documents"]:

        print("\nEVIDENCE:")

        for i, doc in enumerate(
            result["documents"][:3],
            1
        ):

            print(
                f"{i}. "
                f"{doc.metadata.get('product_name')} | "
                f"{doc.metadata.get('section')}"
            )

            print(
                doc.page_content[:300]
            )



QUERY: What is the coverage of Hind Crystel Seal?
STATUS: VALID
MESSAGE: Relevant TDS information found.

EVIDENCE:
1. Hind Crystel Seal | COVERAGE
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.


QUERY: What is the shelf life of Hind Crystel Seal?
STATUS: VALID
MESSAGE: Relevant TDS information found.

EVIDENCE:
1. Hind Crystel Seal | SHELF LIFE
Best before 12 months from the date of manufacture. Should be stored in a cool and dry place
o o
away from direct sunlight at temperature ranging between 2 C (Min.) and 45 C (Max.).


QUERY: What is the packing of Hind Crystal Seal?
STATUS: VALID
MESSAGE: Relevant TDS information found.

EVIDENCE:
1. Hind Crystel Seal | PACKING
10 Kg. in HDPE Bag.


QUERY: What is the coverage?
STATUS: INCOMPLETE_QUERY
MESSAGE: Please specify the product name.


QUERY: What is the coverage of ABC Chemical XYZ?
STATUS: INCOMPLETE_QUERY
MESSAGE: Please specify the product name.


QUERY: Who is the Prime Minister of Ind

In [249]:
queries = [
    "What is the coverage?",
    "What is the coverage of Hind Crystel Seal?",
    "What is the coverage of Hind Crystal Seal?",
    "What is the coverage of ABC Chemical XYZ?"
]

for query in queries:

    print(
        query,
        "→",
        get_product_status(query)
    )

What is the coverage? → {'status': 'NOT_SPECIFIED', 'products': []}
What is the coverage of Hind Crystel Seal? → {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
What is the coverage of Hind Crystal Seal? → {'status': 'FOUND', 'product': {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}}
What is the coverage of ABC Chemical XYZ? → {'status': 'NOT_SPECIFIED', 'products': []}


Phase - 8

In [250]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in .env file")

print("Gemini API key loaded successfully.")

Gemini API key loaded successfully.


In [312]:
from langchain_google_genai import ChatGoogleGenerativeAI

# gemini-3.6-flash

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

In [330]:
from langchain_core.prompts import ChatPromptTemplate

answer_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are the Hindcon TDS Product Knowledge Assistant.

Your ONLY trusted source is the TDS CONTEXT provided below.

STRICT RULES:

1. Answer ONLY from the provided TDS CONTEXT.

2. NEVER use:
   - General knowledge
   - Internet knowledge
   - Your training knowledge
   - Assumptions
   - Guesses
   - Estimated values

3. If the requested information is not clearly present
   in the TDS CONTEXT, respond exactly:

   "The requested information is not available in the provided TDS."

4. Preserve all technical information exactly:
   - Numbers
   - Ranges
   - Units
   - Percentages
   - Temperatures
   - Durations
   - Conditions
   - Minimum/maximum values

5. Do not change the technical meaning of the TDS.

6. Answer only what the user asked.

7. Keep the answer concise and easy to understand.

8. If multiple sources contain the same information,
   prefer the most relevant source.

9. If the user asks for a comparison:
   - Only compare products present in the TDS CONTEXT.
   - Do not assume missing values.
   - If information for one product is missing,
     clearly state that the information is unavailable.

10. Do not invent page numbers.

11. At the end, provide a SOURCE section using only
    metadata supplied in the TDS CONTEXT.

12. The SOURCE section must contain:
    - Product Name
    - Product ID
    - TDS Section
    - Document

13. Never create a source that does not exist in the context.

The TDS CONTEXT is the only trusted source.
"""
    ),
    (
        "human",
        """
USER QUESTION:
{question}

TDS CONTEXT:
{context}

Answer the user's question using ONLY the TDS context.
"""
    )
])

In [329]:
def format_tds_context(documents):
    context_parts = []

    for i, doc in enumerate(documents, 1):

        metadata = doc.metadata

        context_parts.append(
            f"""
SOURCE {i}

Product Name:
{metadata.get("product_name", "Not available")}

Product ID:
{metadata.get("product_id", "Not available")}

Category:
{metadata.get("category", "Not available")}

TDS Section:
{metadata.get("section", "Not available")}

Document:
{metadata.get("document_name", "Not available")}

TDS Content:
{doc.page_content}
"""
        )

    return "\n\n-----------------------------\n\n".join(context_parts)

In [331]:
def format_sources(documents):

    sources = []
    seen_sources = set()

    for doc in documents:

        metadata = doc.metadata

        source_key = (
            metadata.get("product_id"),
            metadata.get("section"),
            metadata.get("document_name")
        )

        if source_key in seen_sources:
            continue

        seen_sources.add(source_key)

        sources.append({
            "product_name": metadata.get("product_name"),
            "product_id": metadata.get("product_id"),
            "section": metadata.get("section"),
            "document_name": metadata.get("document_name"),
            "tds_link": metadata.get("tds_link")
        })

    return sources

In [253]:
query = "What is the coverage of Hind Crystel Seal?"

retrieval = retrieve_with_validation(
    query,
    k=5
)

print("STATUS:", retrieval["status"])

STATUS: VALID


In [254]:
if retrieval["documents"]:

    context = format_tds_context(
        retrieval["documents"][:3]
    )

    print(context)


SOURCE 1

Product Name:
Hind Crystel Seal

Product ID:
HC001

Category:
Waterproofing Compounds

Section:
COVERAGE

Document:
HIND-CRYSTAL-SEAL_TDS-1.pdf

TDS CONTENT:
Brushing – 0.70 -1.20 Kg per Sq. Mtr. per Coat depending on the porosity of the surface.



In [332]:
def answer_from_tds(query: str, k: int = 5):

    # --------------------------------
    # 1. Retrieve and validate
    # --------------------------------

    retrieval = retrieve_with_validation(query, k=k)

    if retrieval["status"] != QUERY_STATUS["VALID"]:

        return {
            "answer": retrieval["message"],
            "status": retrieval["status"],
            "sources": [],
            "evidence_score": retrieval.get("evidence", {}).get("score", 0)
        }

    # --------------------------------
    # 2. Get verified evidence
    # --------------------------------

    documents = retrieval["documents"][:3]

    if not documents:

        return {
            "answer":
                "The requested information is not available in the provided TDS.",
            "status": QUERY_STATUS["INFORMATION_NOT_AVAILABLE"],
            "sources": [],
            "evidence_score": 0
        }

    # --------------------------------
    # 3. Build TDS context
    # --------------------------------

    context = format_tds_context(documents)

    # --------------------------------
    # 4. Send ONLY verified context
    #    to Gemini
    # --------------------------------

    messages = answer_prompt.format_messages(
        question=query,
        context=context
    )

    response = llm.invoke(messages)

    # Gemini 3.x
    answer_text = response.text

    # --------------------------------
    # 5. Format sources
    # --------------------------------

    sources = format_sources(documents)

    # --------------------------------
    # 6. Return final result
    # --------------------------------

    return {
        "answer": answer_text,
        "status": retrieval["status"],
        "sources": sources,
        "evidence_score": retrieval["evidence"]["score"]
    }

In [333]:
result = answer_from_tds(
    "What is the setting time of Hind Plug - S?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
The setting time of Hind Plug - S at 27°C is 1 to 3 minutes.

SOURCE:
Product Name: Hind Plug - S
Product ID: HC003
TDS Section: PROPERTIES
Document: HIND-PLUG-S.pdf

SOURCES:
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'HANDLING PRECAUTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'DESCRIPTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'PROPERTIES', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}


In [335]:
result = answer_from_tds(
    "What is the color of Hind Crystel Seal(1)?"
)

print("STATUS:", result["status"])
print("ANSWER:", result["answer"])

STATUS: VALID
ANSWER: The requested information is not available in the provided TDS.

SOURCE:
Product Name: Hind Crystel Seal (1)
Product ID: HC002
TDS Section: DESCRIPTION, USES, APPLICATION
Document: HIND-CRYSTAL-SEAL-I_TDS.pdf


In [258]:
result = answer_from_tds(
    "What is the coverage?"
)

print("STATUS:", result["status"])
print("ANSWER:", result["answer"])

STATUS: INCOMPLETE_QUERY
ANSWER: Please specify the product name.


In [259]:
result = answer_from_tds(
    "What is the coverage of ABC Chemical XYZ?"
)

print("STATUS:", result["status"])
print("ANSWER:", result["answer"])

STATUS: INCOMPLETE_QUERY
ANSWER: Please specify the product name.


In [260]:
result = answer_from_tds(
    "What is the shelf life of Hind Crystel Seal?"
)

print(result["answer"])

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'The shelf life of Hind Crystel Seal is best before 12 months from the date of manufacture. It should be stored in a cool and dry place away from direct sunlight at a temperature ranging between 2°C (Min.) and 45°C (Max.).\n\nSource: HIND-CRYSTAL-SEAL_TDS-1.pdf', 'extras': {'signature': 'Eq8OCqwOARFNMg/zA+g20QiX7CXcYGtzmolHz11hlIUEyeWdYGnVTLRJ/OrxLaOLTSUTPv+qvPvm/FFY7EvoNsnkGJEKpIsdXQdyYrhkKkXn6Z80IXKdFmDDtBzGLglVSNBs0HNDCQ9/f2ml+9E9koUPfPqQ762Jd4VNdQ31NKa3pXoE9wGVeF63x/f5S1bJ+seZcu3hKltRI35S7K24rXVgDFkllWSO0FYCntmJYkYW2oP707euzjft98ds9SR/ThcL4jfuldbfQiQyRjjZFOecvWDehUPGjHHg3DodzSZuulcKIxYD8f/FAi8vzMmATEHHo3uwnqYkEO+DT3llH6aX90yh1rdzR33ugLw3m2Ml4lyqgMa7/ggt4oEU9b8U0baeHPbNTLr8WLHxyhHEmZkOWTWac25tqtlgKnfGE/NdpCIWMhYhDpjEyt2lXfEYh8+cfH8kyyl+pwyl+FbLHsSppcT8WiUJhGeMlrAZ5BaG0sr3IT6A9+9ScMolGV7cB5YqUwVXCvd9uhnT9yxGpQdPcPtyH+//cApQpvKU9fdvwS+6pfo0AlRvcz6whmCjpRgXqEgJBLLQ15jbmfNkZuPf17XsUxjj4VOoT9xvoXvkdJAerOcmCczGv8uOnEogjQhlhuIPNUxbqBDTNj45YC1uCQmdVSgQpNnMZoSR3dNX7

In [281]:
TECHNICAL_FIELD_ALIASES = {
    "SETTING TIME": [
        "setting time",
        "set time",
        "initial setting",
        "final setting",
        "setting at",
        "setting"
    ],

    "COMPRESSIVE STRENGTH": [
        "compressive strength",
        "compressive"
    ],

    "COVERAGE": [
        "coverage",
        "consumption",
        "application rate",
        "material required"
    ],

    "SHELF LIFE": [
        "shelf life",
        "storage life",
        "expiry",
        "expiration"
    ],

    "PACKING": [
        "packing",
        "pack size",
        "package",
        "packaging"
    ],

    "DOSAGE": [
        "dosage",
        "dose",
        "recommended dosage"
    ],

    "DENSITY": [
        "density",
        "bulk density"
    ],

    "PARTICLE SIZE": [
        "particle size",
        "particle"
    ]
}

In [283]:
def detect_technical_field(query):
    normalized_query = normalize_text(query)

    matches = []

    for field, aliases in TECHNICAL_FIELD_ALIASES.items():

        for alias in aliases:

            if normalize_text(alias) in normalized_query:
                matches.append(field)
                break

    return list(dict.fromkeys(matches))

In [284]:
print(
    detect_technical_field(
        "What is the setting time of Hind Plug - S?"
    )
)

['SETTING TIME']


In [285]:
def technical_field_search(
    query,
    documents,
    k=5,
    product_id=None
):
    technical_fields = detect_technical_field(query)

    if not technical_fields:
        return []

    scored_documents = []

    for doc in documents:

        if product_id:
            if doc.metadata.get("product_id") != product_id:
                continue

        text = normalize_text(doc.page_content)

        if not text:
            continue

        score = 0

        # Check whether requested technical field
        # is actually present in the document
        for field in technical_fields:

            aliases = TECHNICAL_FIELD_ALIASES[field]

            for alias in aliases:

                normalized_alias = normalize_text(alias)

                if normalized_alias in text:

                    # Strong bonus
                    score += 20
                    break

        if score > 0:
            scored_documents.append(
                (score, doc)
            )

    scored_documents.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return [
        doc
        for score, doc in scored_documents[:k]
    ]

In [286]:
setting_docs = technical_field_search(
    "What is the setting time of Hind Plug - S?",
    final_chunks,
    k=5,
    product_id="HC003"
)

for i, doc in enumerate(setting_docs, 1):

    print("\n==============================")
    print("DOCUMENT", i)
    print("PRODUCT:", doc.metadata.get("product_name"))
    print("PRODUCT ID:", doc.metadata.get("product_id"))
    print("SECTION:", doc.metadata.get("section"))

    print("\nCONTENT:")
    print(doc.page_content)


DOCUMENT 1
PRODUCT: Hind Plug - S
PRODUCT ID: HC003
SECTION: DESCRIPTION

CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.

DOCUMENT 2
PRODUCT: Hind Plug - S
PRODUCT ID: HC003
SECTION: PROPERTIES

CONTENT:
Aspect Light Brown Powder
o
Setting at 27 C 1 to 3 minutes
2
Compressive Strength 8 N/mm (minimum) after 1 hour
2
19 N/mm (minimum) after 24 hour

DOCUMENT 3
PRODUCT: Hind Plug - S
PRODUCT ID: HC003
SECTION: DESCRIPTION

CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.

DOCUMENT 4
PRODUCT: Hind Plug - S
PRODUCT ID: HC003
SECTION: PROPERTIES

CONTENT:
Aspect Light Brown Powder
o
Setting at 27 C 1 to 3 minutes
Compressive Strength 8 N/mm2 (minimum) after 1 hour
19 N/mm2 (minimum) after 24 hour

DOCUMENT 5
PRODUCT: Hind Plug - S
PRODUCT ID: HC003
SECTION: DESCRIPTION

CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.


In [287]:
def hybrid_search(
    query,
    k=5,
    product_id=None,
    section=None
):

    # ---------------------------------
    # 1. Semantic search
    # ---------------------------------

    semantic_results = search_tds(
        query,
        k=k,
        product_id=product_id,
        section=section
    )

    # ---------------------------------
    # 2. Keyword search
    # ---------------------------------

    keyword_results = keyword_search(
        query,
        final_chunks,
        k=k,
        product_id=product_id,
        section=section
    )

    # ---------------------------------
    # 3. Technical field search
    # ---------------------------------

    technical_results = technical_field_search(
        query,
        final_chunks,
        k=k,
        product_id=product_id
    )

    # ---------------------------------
    # 4. Combine results
    # ---------------------------------

    combined = (
        technical_results
        + keyword_results
        + semantic_results
    )

    # ---------------------------------
    # 5. Remove duplicates
    # ---------------------------------

    combined = remove_duplicate_chunks(
        combined
    )

    # ---------------------------------
    # 6. Rank evidence
    # ---------------------------------

    ranked = rank_evidence(
        query,
        combined
    )

    return ranked[:k]

In [288]:
def evidence_score(query: str, document):

    query_terms = extract_query_terms(query)

    text = normalize_text(
        document.page_content
    )

    score = 0

    # ---------------------------------
    # 1. Normal keyword matching
    # ---------------------------------

    for term in query_terms:

        if term in text:

            if " " in term:
                score += 3
            else:
                score += 1

    # ---------------------------------
    # 2. Product match
    # ---------------------------------

    product_name = normalize_product_query(
        str(
            document.metadata.get(
                "product_name",
                ""
            )
        )
    )

    normalized_query = normalize_product_query(
        query
    )

    if (
        product_name
        and product_name in normalized_query
    ):
        score += 3

    # ---------------------------------
    # 3. Technical section bonus
    # ---------------------------------

    technical_sections = {
        "PROPERTIES",
        "TECHNICAL DATA",
        "SPECIFICATIONS",
        "COVERAGE",
        "SHELF LIFE",
        "PACKING"
    }

    section = document.metadata.get(
        "section"
    )

    if section in technical_sections:
        score += 2

    # ---------------------------------
    # 4. Exact technical field bonus
    # ---------------------------------

    technical_fields = detect_technical_field(
        query
    )

    for field in technical_fields:

        aliases = TECHNICAL_FIELD_ALIASES[field]

        for alias in aliases:

            normalized_alias = normalize_text(
                alias
            )

            if normalized_alias in text:

                # VERY strong evidence
                score += 20
                break

    return score

In [289]:
setting_docs = hybrid_search(
    "What is the setting time of Hind Plug - S?",
    k=5,
    product_id="HC003"
)

for i, doc in enumerate(setting_docs, 1):

    print("\n==============================")
    print("DOCUMENT", i)

    print(
        "SECTION:",
        doc.metadata.get("section")
    )

    print(
        "SCORE:",
        evidence_score(
            "What is the setting time of Hind Plug - S?",
            doc
        )
    )

    print("\nCONTENT:")
    print(doc.page_content)


DOCUMENT 1
SECTION: HANDLING PRECAUTION
SCORE: 37

CONTENT:
information or recommendations complete, accurate or useful. The client must test and ascertain the safety and ﬁtness for the product for use. The right to change the
properties of the products is reserved with us. The proprietary rights of third parties must be observed. All orders are accepted subject to the terms of sale and delivery.
Users must always refer to the most recent issue of the latest Data Sheet for the product concerned, copies of which will be supplied on request.
IMPORTANT NOTICE :
HINDCONCHEMICALS LTD.
VISIT OUR WEBSITE
HIND PLUG - S
A QUICK SETTING READY TO USE POWDERED COMPOUND

DOCUMENT 2
SECTION: DESCRIPTION
SCORE: 30

CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.

DOCUMENT 3
SECTION: PROPERTIES
SCORE: 27

CONTENT:
Aspect Light Brown Powder
o
Setting at 27 C 1 to 3 minutes
2
Compressive Strength 8 N/mm (minimum) after 1 hour
2
19 N/mm (minimum) after 24 hour

DOCUMEN

Phase - 9

In [300]:
def answer_from_tds(query: str, k: int = 5):
    retrieval = retrieve_with_validation(query, k=k)

    # Handle invalid / incomplete / not-found queries
    if retrieval["status"] != QUERY_STATUS["VALID"]:
        evidence_score = 0

        if "evidence" in retrieval:
            evidence_score = retrieval["evidence"].get("score", 0)

        return {
            "answer": retrieval["message"],
            "status": retrieval["status"],
            "sources": [],
            "evidence_score": evidence_score
        }

    # Use only the strongest evidence
    documents = retrieval["documents"][:3]

    context = format_tds_context(documents)

    messages = answer_prompt.format_messages(
        question=query,
        context=context
    )

    response = llm.invoke(messages)

    # Gemini 3.6 structured response
    answer_text = response.text

    # Build source information
    sources = []
    seen_sources = set()

    for doc in documents:
        metadata = doc.metadata

        source_key = (
            metadata.get("product_id"),
            metadata.get("section"),
            metadata.get("document_name")
        )

        if source_key in seen_sources:
            continue

        seen_sources.add(source_key)

        sources.append({
            "product_name": metadata.get("product_name"),
            "product_id": metadata.get("product_id"),
            "section": metadata.get("section"),
            "document_name": metadata.get("document_name"),
            "tds_link": metadata.get("tds_link")
        })

    return {
        "answer": answer_text,
        "status": retrieval["status"],
        "sources": sources,
        "evidence_score": retrieval["evidence"].get("score", 0)
    }

In [301]:
result = answer_from_tds(
    "What is the shelf life of Hind Crystel Seal?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


STATUS:
VALID

ANSWER:
The shelf life of Hind Crystel Seal is best before 12 months from the date of manufacture. It should be stored in a cool and dry place away from direct sunlight at a temperature ranging between 2 C (Min.) and 45 C (Max.).

**Source:**
* **Product Name:** Hind Crystel Seal
* **Document:** HIND-CRYSTAL-SEAL_TDS-1.pdf

SOURCES:
{'product_name': 'Hind Crystel Seal', 'product_id': 'HC001', 'section': 'SHELF LIFE', 'document_name': 'HIND-CRYSTAL-SEAL_TDS-1.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL_TDS-1.pdf'}


In [302]:
result = answer_from_tds(
    "What is the setting time of Hind Plug - S?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


STATUS:
VALID

ANSWER:
The setting time of Hind Plug - S at 27 C is 1 to 3 minutes.

**Source:**
- Product Name: Hind Plug - S
- Document: HIND-PLUG-S.pdf

SOURCES:
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'HANDLING PRECAUTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'DESCRIPTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'PROPERTIES', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}


In [293]:
test_queries = [
    "What is the shelf life of Hind Crystel Seal?",
    "What is the setting time of Hind Plug - S?",
    "What is the compressive strength of Hind Plug - S?"
]

for query in test_queries:

    print("\n" + "=" * 70)
    print("QUESTION:", query)

    result = answer_from_tds(query)

    print("STATUS:", result["status"])
    print("ANSWER:", result["answer"])

    print("SOURCES:")
    for source in result["sources"]:
        print(source)


QUESTION: What is the shelf life of Hind Crystel Seal?


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


STATUS: VALID
ANSWER: The shelf life of Hind Crystel Seal is best before 12 months from the date of manufacture. It should be stored in a cool and dry place away from direct sunlight at a temperature ranging between 2 C (Min.) and 45 C (Max.).

**Source:**
- **Product Name:** Hind Crystel Seal (Product ID: HC001)
- **Document:** HIND-CRYSTAL-SEAL_TDS-1.pdf
SOURCES:
{'product_name': 'Hind Crystel Seal', 'product_id': 'HC001', 'category': 'Waterproofing Compounds', 'section': 'SHELF LIFE', 'document_name': 'HIND-CRYSTAL-SEAL_TDS-1.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL_TDS-1.pdf'}

QUESTION: What is the setting time of Hind Plug - S?


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


STATUS: VALID
ANSWER: The setting time of Hind Plug - S at 27°C is 1 to 3 minutes.

**Source:**
- **Product Name:** Hind Plug - S
- **Document:** HIND-PLUG-S.pdf
- **Section:** PROPERTIES
SOURCES:
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'category': 'Waterproofing Compounds', 'section': 'HANDLING PRECAUTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'category': 'Waterproofing Compounds', 'section': 'DESCRIPTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'category': 'Waterproofing Compounds', 'section': 'PROPERTIES', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}

QUESTION: What is the compressive strength of Hind Plug - S?


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\langchain_google_genai\chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


STATUS: VALID
ANSWER: The compressive strength of Hind Plug - S is:

* **8 N/mm² (minimum)** after 1 hour
* **19 N/mm² (minimum)** after 24 hours

**Source:**
* **Product Name:** Hind Plug - S
* **Document:** HIND-PLUG-S.pdf
SOURCES:
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'category': 'Waterproofing Compounds', 'section': 'PROPERTIES', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}


In [309]:
results = technical_field_search(
    "What is the setting time of Hind Plug - S?",
    final_chunks,
    k=5,
    product_id="HC003"
)

for i, doc in enumerate(results, 1):
    print("\n==============================")
    print("RESULT:", i)
    print("SECTION:", doc.metadata.get("section"))
    print("PRODUCT:", doc.metadata.get("product_name"))
    print("CONTENT:")
    print(doc.page_content)


RESULT: 1
SECTION: DESCRIPTION
PRODUCT: Hind Plug - S
CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.

RESULT: 2
SECTION: PROPERTIES
PRODUCT: Hind Plug - S
CONTENT:
Aspect Light Brown Powder
o
Setting at 27 C 1 to 3 minutes
2
Compressive Strength 8 N/mm (minimum) after 1 hour
2
19 N/mm (minimum) after 24 hour

RESULT: 3
SECTION: DESCRIPTION
PRODUCT: Hind Plug - S
CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.

RESULT: 4
SECTION: PROPERTIES
PRODUCT: Hind Plug - S
CONTENT:
Aspect Light Brown Powder
o
Setting at 27 C 1 to 3 minutes
Compressive Strength 8 N/mm2 (minimum) after 1 hour
19 N/mm2 (minimum) after 24 hour

RESULT: 5
SECTION: DESCRIPTION
PRODUCT: Hind Plug - S
CONTENT:
Hind Plug S is fast setting powdered compound for stopping running water.


In [307]:
query = "What is the dosage of Hind Crystel Seal (1)"

print("QUERY:")
print(query)

print("\nNORMALIZED QUERY:")
print(normalize_product_query(query))

print("\nDETECTED PRODUCT:")
print(detect_product(query))

print("\nPRODUCT LOOKUP MATCHES:")
normalized_query = normalize_product_query(query)

for name, info in product_lookup.items():
    if "crystel seal" in name:
        print("LOOKUP NAME:", name)
        print("PRODUCT INFO:", info)

QUERY:
What is the dosage of Hind Crystel Seal (1)

NORMALIZED QUERY:
what is the dosage of hind crystel seal 1

DETECTED PRODUCT:
{'status': 'FOUND', 'product': {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}}

PRODUCT LOOKUP MATCHES:
LOOKUP NAME: hind crystel seal
PRODUCT INFO: {'product_id': 'HC001', 'product_name': 'Hind Crystel Seal', 'category': 'Waterproofing Compounds'}
LOOKUP NAME: hind crystel seal 1
PRODUCT INFO: {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}


In [310]:
query = "What is the dosage of Hind Crystel Seal (1)"

print("QUERY:")
print(query)

print("\nNORMALIZED QUERY:")
print(normalize_product_query(query))

print("\nDETECTED PRODUCT:")
print(detect_product(query))

QUERY:
What is the dosage of Hind Crystel Seal (1)

NORMALIZED QUERY:
what is the dosage of hind crystel seal 1

DETECTED PRODUCT:
{'status': 'FOUND', 'product': {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}}


In [314]:
result = answer_from_tds(
    "What is the minimum compressive strength of Hind Plug - S after 24 hours?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
The minimum compressive strength of Hind Plug - S after 24 hours is 19 N/mm2.

Source: HIND-PLUG-S.pdf

SOURCES:
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'PROPERTIES', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}


In [315]:
result = answer_from_tds(
    "WTell me detailed about Hind Acrylic Wall Putty"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
Hind Acrylic Wall Putty (HC080) is an acrylic-based, ready-to-use, water-based wall putty designed for interior masonry walls. It is used for filling and smoothing out cracks, undulations, and dents on plastered masonry walls or similar interior surfaces.

**Key Properties:**
*   **Consistency:** Paste (Smooth), described as butter-like for easy application, filling, leveling, and sanding.
*   **Aspect:** White only.
*   **Density:** 1.65 - 1.75.
*   **Volatile Organic Content (VOC):** 25 gm/Liter (Theoretical).
*   **Surface Finish:** Smooth and Matt.
*   **Number of Coats Required:** 1-2 coats.
*   **Primer/Undercoat:** Cement Primer.

**Application Details:**
*   **Method of Application:** Apply with a putty knife.
*   **Thinner:** Thinning is not recommended; however, if required, potable water can be added.
*   **Drying Time (@ 28°C to 30°C and 60±5% Relative Humidity):**
    *   Touch Dry: Not more than 30 minutes.
    *   Hard Dry: Not more than 4 hours.



In [320]:
result = answer_from_tds(
    "What is the coverage range of Hind Crystel Seal?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
The coverage of Hind Crystel Seal is 0.70 - 1.20 Kg per Sq. Mtr. per coat, depending on the porosity of the surface.

Source: HIND-CRYSTAL-SEAL_TDS-1.pdf

SOURCES:
{'product_name': 'Hind Crystel Seal', 'product_id': 'HC001', 'section': 'COVERAGE', 'document_name': 'HIND-CRYSTAL-SEAL_TDS-1.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL_TDS-1.pdf'}


In [322]:
result = answer_from_tds(
    "What is the coverage of Hind Crystal Seal?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
The coverage of Hind Crystel Seal is 0.70 - 1.20 Kg per Sq. Mtr. per coat, depending on the porosity of the surface.

Source: HIND-CRYSTAL-SEAL_TDS-1.pdf

SOURCES:
{'product_name': 'Hind Crystel Seal', 'product_id': 'HC001', 'section': 'COVERAGE', 'document_name': 'HIND-CRYSTAL-SEAL_TDS-1.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL_TDS-1.pdf'}


In [323]:
result = answer_from_tds(
    "What is the color of Hind Plug - S?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
The requested information is not available in the provided TDS.

Source: HIND-PLUG-S.pdf

SOURCES:
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'GENERAL', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}
{'product_name': 'Hind Plug - S', 'product_id': 'HC003', 'section': 'HANDLING PRECAUTION', 'document_name': 'HIND-PLUG-S.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-PLUG-S.pdf'}


In [324]:
result = answer_from_tds(
    "What is the coverage?"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")
for source in result["sources"]:
    print(source)

STATUS:
INCOMPLETE_QUERY

ANSWER:
Please specify the product name.

SOURCES:


Phase - 9

In [325]:
RESPONSE_MESSAGES = {
    "INCOMPLETE_QUERY":
        "Please specify which product you mean.",

    "PRODUCT_NOT_FOUND":
        "The specified product was not found in the Hindcon TDS database.",

    "INFORMATION_NOT_AVAILABLE":
        "The requested information is not available in the provided TDS.",

    "INVALID_QUERY":
        "Invalid question. Please ask a question related to a Hindcon product or its TDS.",

    "AMBIGUOUS_PRODUCT":
        "Multiple products match your question. Please specify the exact product name."
}

In [326]:
def validate_query(query: str):
    """
    Validate the user's query before retrieval.
    """

    # Basic validation
    if not isinstance(query, str):
        return {
            "status": QUERY_STATUS["INVALID_QUERY"],
            "message": RESPONSE_MESSAGES["INVALID_QUERY"]
        }

    query = query.strip()

    if not query:
        return {
            "status": QUERY_STATUS["INVALID_QUERY"],
            "message": RESPONSE_MESSAGES["INVALID_QUERY"]
        }

    if len(query) < 3:
        return {
            "status": QUERY_STATUS["INVALID_QUERY"],
            "message": RESPONSE_MESSAGES["INVALID_QUERY"]
        }

    # Product detection
    product_status = get_product_status(query)

    # Product explicitly specified but not found
    if product_status["status"] == "PRODUCT_NOT_FOUND":
        return {
            "status": QUERY_STATUS["PRODUCT_NOT_FOUND"],
            "message": RESPONSE_MESSAGES["PRODUCT_NOT_FOUND"]
        }

    # Multiple products detected
    if product_status["status"] == "AMBIGUOUS":
        return {
            "status": QUERY_STATUS["INCOMPLETE_QUERY"],
            "message": RESPONSE_MESSAGES["AMBIGUOUS_PRODUCT"]
        }

    # No product specified
    if product_status["status"] == "NOT_SPECIFIED":
        return {
            "status": QUERY_STATUS["INCOMPLETE_QUERY"],
            "message": RESPONSE_MESSAGES["INCOMPLETE_QUERY"]
        }

    # Product found
    return {
        "status": QUERY_STATUS["VALID"],
        "message": "Query is valid.",
        "product": product_status["product"]
    }

In [327]:
test_queries = [
    "",
    "hi",
    "What is the dosage?",
    "What is the dosage of Unknown Product XYZ?",
    "What is the dosage of Hind Crystel Seal (1)"
]

for query in test_queries:

    result = validate_query(query)

    print("=" * 70)
    print("QUERY:", query)
    print("STATUS:", result["status"])
    print("MESSAGE:", result["message"])

    if "product" in result:
        print("PRODUCT:", result["product"])

QUERY: 
STATUS: INVALID_QUERY
MESSAGE: Invalid question. Please ask a question related to a Hindcon product or its TDS.
QUERY: hi
STATUS: INVALID_QUERY
MESSAGE: Invalid question. Please ask a question related to a Hindcon product or its TDS.
QUERY: What is the dosage?
STATUS: INCOMPLETE_QUERY
MESSAGE: Please specify which product you mean.
QUERY: What is the dosage of Unknown Product XYZ?
STATUS: INCOMPLETE_QUERY
MESSAGE: Please specify which product you mean.
QUERY: What is the dosage of Hind Crystel Seal (1)
STATUS: VALID
MESSAGE: Query is valid.
PRODUCT: {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}


In [336]:
result = answer_from_tds(
    "What is the dosage of Hind Crystel Seal (1)"
)

print(result["status"])
print(result["answer"])

VALID
The dosage of Hind Crystel Seal (1) is 0.8 to 1.0% by weight of Cement/Cementitious binder.

SOURCE:
- Product Name: Hind Crystel Seal (1)
- Product ID: HC002
- TDS Section: APPLICATION
- Document: HIND-CRYSTAL-SEAL-I_TDS.pdf


In [337]:
result = answer_from_tds(
    "What is the dosage of Hind Crystel Seal (1)"
)

print("STATUS:")
print(result["status"])

print("\nANSWER:")
print(result["answer"])

print("\nSOURCES:")

for source in result["sources"]:
    print(source)

STATUS:
VALID

ANSWER:
The dosage of Hind Crystel Seal (1) is 0.8 to 1.0% by weight of Cement/Cementitious binder.

SOURCE:
- Product Name: Hind Crystel Seal (1)
- Product ID: HC002
- TDS Section: APPLICATION
- Document: HIND-CRYSTAL-SEAL-I_TDS.pdf

SOURCES:
{'product_name': 'Hind Crystel Seal (1)', 'product_id': 'HC002', 'section': 'APPLICATION', 'document_name': 'HIND-CRYSTAL-SEAL-I_TDS.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL-I_TDS.pdf'}
{'product_name': 'Hind Crystel Seal (1)', 'product_id': 'HC002', 'section': 'ADVANTAGES', 'document_name': 'HIND-CRYSTAL-SEAL-I_TDS.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL-I_TDS.pdf'}


Phase - 11

In [342]:
conversation_state = {
    "product": None,
    "product_id": None,
    "product_name": None,
    "category": None
}

In [343]:
def get_product_with_context(query: str, state: dict):

    # First try to detect product from current query
    product_status = get_product_status(query)

    # Product explicitly found
    if product_status["status"] == "FOUND":

        product = product_status["product"]

        return {
            "status": "FOUND",
            "product": product,
            "source": "CURRENT_QUERY"
        }

    # Product explicitly ambiguous
    if product_status["status"] == "AMBIGUOUS":

        return {
            "status": "AMBIGUOUS",
            "products": product_status["products"],
            "source": "CURRENT_QUERY"
        }

    # Product looks like an unknown product
    if product_status["status"] == "PRODUCT_NOT_FOUND":

        return {
            "status": "PRODUCT_NOT_FOUND",
            "products": [],
            "source": "CURRENT_QUERY"
        }

    # No product in current query
    # Try previous conversation context
    if state.get("product_id"):

        return {
            "status": "FOUND",
            "product": {
                "product_id": state["product_id"],
                "product_name": state["product_name"],
                "category": state["category"]
            },
            "source": "CONVERSATION_CONTEXT"
        }

    # Nothing available
    return {
        "status": "NOT_SPECIFIED",
        "products": [],
        "source": "NONE"
    }

In [344]:
def update_conversation_state(state: dict, product: dict):

    state["product"] = product
    state["product_id"] = product["product_id"]
    state["product_name"] = product["product_name"]
    state["category"] = product["category"]

    return state

In [345]:
def route_query_with_context(query: str, state: dict):

    if is_obviously_invalid_query(query):

        return {
            "status": QUERY_STATUS["INVALID_QUERY"],
            "message": RESPONSE_MESSAGES["INVALID_QUERY"]
        }

    product_result = get_product_with_context(query, state)

    # Unknown product
    if product_result["status"] == "PRODUCT_NOT_FOUND":

        return {
            "status": QUERY_STATUS["PRODUCT_NOT_FOUND"],
            "message": RESPONSE_MESSAGES["PRODUCT_NOT_FOUND"]
        }

    # Ambiguous product
    if product_result["status"] == "AMBIGUOUS":

        return {
            "status": QUERY_STATUS["INCOMPLETE_QUERY"],
            "message": RESPONSE_MESSAGES["AMBIGUOUS_PRODUCT"]
        }

    # No product
    if product_result["status"] == "NOT_SPECIFIED":

        return {
            "status": QUERY_STATUS["INCOMPLETE_QUERY"],
            "message": RESPONSE_MESSAGES["INCOMPLETE_QUERY"]
        }

    # Product found
    product = product_result["product"]

    # Save product in conversation
    update_conversation_state(state, product)

    # Detect section
    section_result = detect_section(query)

    return {
        "status": QUERY_STATUS["VALID"],
        "product": product,
        "section_detection": section_result,
        "product_source": product_result["source"]
    }

In [346]:
conversation_state = {
    "product": None,
    "product_id": None,
    "product_name": None,
    "category": None
}

In [347]:
query1 = "What is the dosage of Hind Crystel Seal (1)"

result1 = route_query_with_context(
    query1,
    conversation_state
)

print(result1)
print("\nSTATE:")
print(conversation_state)

{'status': 'VALID', 'product': {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}, 'section_detection': {'status': 'NOT_FOUND', 'sections': []}, 'product_source': 'CURRENT_QUERY'}

STATE:
{'product': {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}, 'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}


In [348]:
query2 = "What is its shelf life?"

result2 = route_query_with_context(
    query2,
    conversation_state
)

print(result2)

{'status': 'VALID', 'product': {'product_id': 'HC002', 'product_name': 'Hind Crystel Seal (1)', 'category': 'Waterproofing Compounds'}, 'section_detection': {'status': 'FOUND', 'section': 'SHELF LIFE'}, 'product_source': 'CONVERSATION_CONTEXT'}


In [349]:
def build_contextual_query(query: str, product: dict):

    product_name = product["product_name"]

    normalized_query = normalize_text(query)

    contextual_words = [
        "it",
        "its",
        "this product",
        "that product",
        "the product"
    ]

    contains_reference = any(
        word in normalized_query
        for word in contextual_words
    )

    if contains_reference:

        return f"{query} for {product_name}"

    return query

In [350]:
product = conversation_state["product"]

contextual_query = build_contextual_query(
    "What is its shelf life?",
    product
)

print(contextual_query)

What is its shelf life? for Hind Crystel Seal (1)


In [351]:
def retrieve_with_validation_context(
    query: str,
    state: dict,
    k: int = 5
):

    routing = route_query_with_context(
        query,
        state
    )

    if routing["status"] != QUERY_STATUS["VALID"]:

        return {
            "status": routing["status"],
            "message": routing["message"],
            "documents": [],
            "evidence": {
                "score": 0
            }
        }

    product = routing["product"]

    product_id = product["product_id"]

    contextual_query = build_contextual_query(
        query,
        product
    )

    section_detection = routing["section_detection"]

    section = None

    if section_detection["status"] == "FOUND":
        section = section_detection["section"]

    documents = hybrid_search(
        contextual_query,
        k=k,
        product_id=product_id,
        section=section
    )

    evidence = check_evidence(
        contextual_query,
        documents
    )

    if not evidence["sufficient"]:

        return {
            "status":
                QUERY_STATUS["INFORMATION_NOT_AVAILABLE"],

            "message":
                RESPONSE_MESSAGES[
                    "INFORMATION_NOT_AVAILABLE"
                ],

            "documents": [],

            "evidence": evidence
        }

    return {
        "status": QUERY_STATUS["VALID"],
        "message": "Query is valid.",
        "documents": documents,
        "evidence": evidence,
        "product": product
    }

In [352]:
def answer_from_tds(
    query: str,
    state: dict,
    k: int = 5
):

    retrieval = retrieve_with_validation_context(
        query,
        state,
        k=k
    )

    if retrieval["status"] != QUERY_STATUS["VALID"]:

        return {
            "answer": retrieval["message"],
            "status": retrieval["status"],
            "sources": [],
            "evidence_score": retrieval.get(
                "evidence",
                {}
            ).get(
                "score",
                0
            )
        }

    documents = retrieval["documents"][:3]

    if not documents:

        return {
            "answer":
                "The requested information is not available "
                "in the provided TDS.",

            "status":
                QUERY_STATUS["INFORMATION_NOT_AVAILABLE"],

            "sources": [],

            "evidence_score": 0
        }

    context = format_tds_context(documents)

    messages = answer_prompt.format_messages(
        question=query,
        context=context
    )

    response = llm.invoke(messages)

    answer_text = response.text

    sources = format_sources(documents)

    return {
        "answer": answer_text,
        "status": retrieval["status"],
        "sources": sources,
        "evidence_score":
            retrieval["evidence"]["score"]
    }

In [353]:
state = {
    "product": None,
    "product_id": None,
    "product_name": None,
    "category": None
}

In [354]:
result = answer_from_tds(
    "What is the dosage of Hind Crystel Seal (1)?",
    state=state
)

print(result["answer"])

The dosage of Hind Crystel Seal (1) is 0.8 to 1.0% by weight of Cement/Cementitious binder.

SOURCE:
- Product Name: Hind Crystel Seal (1)
- Product ID: HC002
- TDS Section: APPLICATION
- Document: HIND-CRYSTAL-SEAL-I_TDS.pdf


In [355]:
result = answer_from_tds(
    "What is its shelf life?",
    state=state
)

print(result["answer"])

The shelf life is 12 months from the date of manufacture.

SOURCE:
Product Name: Hind Crystel Seal (1)
Product ID: HC002
TDS Section: SHELF LIFE
Document: HIND-CRYSTAL-SEAL-I_TDS.pdf


In [356]:
state = {
    "product": None,
    "product_id": None,
    "product_name": None,
    "category": None
}

result = answer_from_tds(
    "What is the dosage of Hind Crystel Seal (1)?",
    state=state
)

print("ANSWER:")
print(result["answer"])

print("\nSTATUS:")
print(result["status"])

print("\nSOURCE:")
print(result["sources"])

print("\nEVIDENCE SCORE:")
print(result["evidence_score"])

ANSWER:
The dosage of Hind Crystel Seal (1) is 0.8 to 1.0% by weight of Cement/Cementitious binder.

SOURCE:
- Product Name: Hind Crystel Seal (1)
- Product ID: HC002
- TDS Section: APPLICATION
- Document: HIND-CRYSTAL-SEAL-I_TDS.pdf

STATUS:
VALID

SOURCE:
[{'product_name': 'Hind Crystel Seal (1)', 'product_id': 'HC002', 'section': 'APPLICATION', 'document_name': 'HIND-CRYSTAL-SEAL-I_TDS.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL-I_TDS.pdf'}, {'product_name': 'Hind Crystel Seal (1)', 'product_id': 'HC002', 'section': 'ADVANTAGES', 'document_name': 'HIND-CRYSTAL-SEAL-I_TDS.pdf', 'tds_link': 'https://www.hindcon.com/wp-content/uploads/2025/02/HIND-CRYSTAL-SEAL-I_TDS.pdf'}]

EVIDENCE SCORE:
27


In [357]:
result = answer_from_tds(
    "What is its shelf life?",
    state=state
)

print("ANSWER:")
print(result["answer"])

print("\nSTATUS:")
print(result["status"])

print("\nCURRENT PRODUCT:")
print(state["product_name"])

ANSWER:
The shelf life is 12 months from the date of manufacture.

SOURCE:
Product Name: Hind Crystel Seal (1)
Product ID: HC002
TDS Section: SHELF LIFE
Document: HIND-CRYSTAL-SEAL-I_TDS.pdf

STATUS:
VALID

CURRENT PRODUCT:
Hind Crystel Seal (1)


In [358]:
result = answer_from_tds(
    "What is the setting time of Hind Plug - S?",
    state=state
)

print("ANSWER:")
print(result["answer"])

print("\nCURRENT PRODUCT:")
print(state["product_name"])

ANSWER:
The setting time of Hind Plug - S at 27°C is 1 to 3 minutes.

SOURCE:
Product Name: Hind Plug - S
Product ID: HC003
TDS Section: PROPERTIES
Document: HIND-PLUG-S.pdf

CURRENT PRODUCT:
Hind Plug - S


In [359]:
result = answer_from_tds(
    "What is its compressive strength?",
    state=state
)

print(result["answer"])

The compressive strength of Hind Plug - S is:
* 8 N/mm² (minimum) after 1 hour
* 19 N/mm² (minimum) after 24 hours

SOURCE:
Product Name: Hind Plug - S
Product ID: HC003
TDS Section: PROPERTIES
Document: HIND-PLUG-S.pdf


In [369]:
result = answer_from_tds(
    "tell me detailed about Hind Sol SR this product with the tds pdf link",
    state=state
)

print(result["answer"])

Hind Sol SR is a water-based, medium viscous, liquid surface retarder designed to obtain an exposed aggregate finish on concrete by retarding the setting process of the cement paste on the top surface layer. It can be applied to formwork or sprayed on freshly laid concrete and is ideal for use in tropical and hot climate conditions.

**Properties:**
*   **Aspect:** Dark Brown Liquid
*   **Specific Gravity:** 1.12 + 0.02 at 25°C
*   **pH:** > 6
*   **Chloride Content:** < 0.02%

**Coverage:**
*   10-15 Sq. Mtr/Ltr for up to 8mm depth of retardation.
*   6-8 Sq. Mtr/Ltr for greater depth of retardation.

**Handling Precaution:**
Hind Sol SR is non-toxic, but ingestion must be avoided. In case of skin contact, wash immediately with water. If contact with eyes occurs, wash immediately with water and seek medical advice.

**TDS Document Link:**
HIND-SOL-SR.pdf

***

**SOURCE**
*   **Product Name:** Hind Sol SR
*   **Product ID:** HC257
*   **TDS Section:** DESCRIPTION, PROPERTIES, HANDLING 